# MICA benchmark: pilot of ModelAngelo on entry 26DK

ModelAngelo (v1.0.18) builds an atomic model from the map and the protein sequence alone. This notebook runs it once, on one small entry, to find out how long it
takes on a free GPU, what it writes, and whether the scoring reads its files. No Phenix, no secrets and no private files are needed.

1. checks the machine (a GPU is needed) and the internet,
2. builds ModelAngelo's Python 3.11 environment from exact pins and tests the GPU,
3. fetches ModelAngelo (pinned commit) and its weights (the MD5 is checked by ModelAngelo itself),
4. downloads the map and the deposited model, writes the FASTA file of the entry,
5. runs `model_angelo build` (a progress line every two minutes), then scores the models with US-align and packs one zip.

**What you do (Kaggle, background mode)**: accelerator **GPU T4 x2** (not P100), Internet **On**, then *Save Version -> Save & Run All (Commit)*. It runs on Kaggle's
servers; you can close the tab and your Wi-Fi does not matter. About 40 minutes to 1.5 hours. When the version shows *Succeeded*, open it and download the zip
`modelangelo_pilot_result_....zip` from its Output. Send it to me. **Do not stop or cancel the run** unless it is stuck for more than 3 hours.


## Step 1. Look at the machine

In [ ]:
import csv, datetime, getpass, glob, gzip, hashlib, json, os, platform, re, shutil, signal, subprocess, sys, tarfile, threading, time, zipfile
from collections import deque

IN_KAGGLE = os.path.isdir("/kaggle/working") and not os.environ.get("MICA_WORK")      # MICA_WORK and MICA_BIG are only used by the tests
IN_COLAB = (not IN_KAGGLE) and os.path.isdir("/content") and not os.environ.get("MICA_WORK")
PLATFORM = "kaggle" if IN_KAGGLE else ("colab" if IN_COLAB else "other")
if os.environ.get("MICA_WORK"):
    WORK = OUT_DIR = os.environ["MICA_WORK"]
elif IN_KAGGLE:
    WORK, OUT_DIR = "/kaggle/working/pilot", "/kaggle/working"
elif IN_COLAB:
    WORK, OUT_DIR = "/content/pilot", "/content"
else:
    WORK = OUT_DIR = os.path.abspath("pilot")
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
LOG = os.path.join(WORK, "run.log")
REQUIRE_GPU = True
NET_HOSTS = (('PyPI', 'https://pypi.org/simple/uv/', True), ('GitHub', 'https://github.com/3dem/model-angelo', True), ('Zenodo', 'https://zenodo.org/api/records/7942241', True), ('Meta ESM weights', 'https://dl.fbaipublicfiles.com/fair-esm/models/esm1b_t33_650M_UR50S.pt', True), ('EBI FTP', 'https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/header/emd-80558.xml', False), ('RCSB', 'https://files.rcsb.org/header/26DK.cif', False))
STATE = {"ok": True, "problem": "", "t0": time.time(), "started": datetime.datetime.now(datetime.timezone.utc).isoformat(),
         "phenix_env": None, "phenix_version": "", "installer_sha256": "", "usalign": "", "usalign_version": "", "weights": {}, "env": {},
         "entries": {}, "gpu": ""}
SECRETS = []          # kept in memory only, to make sure they never end up in the zip


def log(msg=""):
    print(msg, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(str(msg) + "\n")


def fail(msg):
    STATE["ok"] = False
    STATE["problem"] = msg
    log("\n" + "!" * 72 + "\nSTOPPED: " + msg + "\nThe steps below are skipped. The last step still packs a zip with the log: send it to me.\n" + "!" * 72)


def run(cmd, env=None, every=1, cwd=None, log_file=None, timeout=None, quiet=False):
    """Runs a command, prints and logs its output as it comes. Returns (exit code, last lines, timed_out).
    env: values are set, None removes the variable. A command that runs longer than `timeout` seconds is killed with its child processes.
    Progress bars (lines with %|) are printed only now and then; quiet=True keeps the output out of the notebook (it still goes to log_file)."""
    e = dict(os.environ)
    for k, v in (env or {}).items():
        if v is None:
            e.pop(k, None)
        else:
            e[k] = v
    e["PYTHONUNBUFFERED"] = "1"
    try:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=e, cwd=cwd, errors="replace",
                             start_new_session=True)
    except OSError as ex:
        log("could not start %s: %s" % (cmd[0] if isinstance(cmd, (list, tuple)) else cmd, ex))
        return 127, [str(ex)], False
    timed = {"out": False}

    def killer():
        timed["out"] = True
        try:
            os.killpg(p.pid, signal.SIGKILL)
        except OSError:
            pass

    timer = threading.Timer(timeout, killer) if timeout else None
    if timer:
        timer.start()
    lf = open(log_file, "a", encoding="utf-8") if log_file else None
    last, n, bars = deque(maxlen=40), 0, 0
    try:
        for line in p.stdout:
            line = line.rstrip("\n")
            if len(line) > 600:
                line = line[:600] + " ..."
            last.append(line)
            n += 1
            if lf:
                lf.write(line + "\n")
                lf.flush()                                  # a program that is silent for an hour must not hide its last lines in a buffer
            if "%|" in line:
                bars += 1
                show = bars % 100 == 1
            else:
                show = every == 1 or n <= 5 or n % every == 0
            if show and not quiet:
                log(line)
        p.wait()
    finally:
        if timer:
            timer.cancel()
        if lf:
            lf.close()
        p.stdout.close()
    return p.returncode, list(last), timed["out"]


def sh(cmd, timeout=60):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout, errors="replace")
        return (r.stdout + r.stderr).strip()
    except Exception as ex:
        return "ERROR: %s" % ex


def secret(name, prompt, hidden=True):
    """Colab Secrets, Kaggle Secrets, an environment variable, or a prompt, in this order. Returns "" if nothing can be found."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(name)
        if v:
            return v.strip()
    except Exception:
        pass
    if os.environ.get(name):
        return os.environ[name].strip()
    try:
        return (getpass.getpass(prompt) if hidden else input(prompt)).strip()
    except Exception:
        return ""


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def md5_file(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def meminfo_gb():
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemTotal"):
                    return round(int(line.split()[1]) / 1048576, 1)
    except OSError:
        pass
    return None


def pick_big_dir():
    """The folder for everything large (Phenix, Python environment, MICA): the disk with the most free space."""
    if os.environ.get("MICA_BIG"):
        return os.environ["MICA_BIG"]
    best = None
    for base in ("/kaggle/temp", "/tmp", "/content", WORK):
        if os.path.isdir(base) and os.access(base, os.W_OK):
            free = shutil.disk_usage(base).free
            if best is None or free > best[1] + 5e9:
                best = (base, free)
    return os.path.join(best[0], "mica_pilot")


BIG = pick_big_dir()
os.makedirs(BIG, exist_ok=True)


class Monitor(threading.Thread):
    """Samples GPU memory, GPU load and free RAM every few seconds into a csv file; summary() gives the peaks."""

    def __init__(self, path, every=5):
        super().__init__(daemon=True)
        self.path, self.every, self.stop_flag = path, every, threading.Event()
        self.peak_gpu_mb, self.peak_util, self.min_ram_gb, self.samples = 0, 0, None, 0

    def sample(self):
        gpu_mb, util = "", ""
        if shutil.which("nvidia-smi"):
            out = sh("nvidia-smi --query-gpu=memory.used,utilization.gpu --format=csv,noheader,nounits", timeout=20).splitlines()
            try:
                a, b = [x.strip() for x in out[0].split(",")]
                gpu_mb, util = int(a), int(b)
                self.peak_gpu_mb, self.peak_util = max(self.peak_gpu_mb, gpu_mb), max(self.peak_util, util)
            except (ValueError, IndexError):
                gpu_mb, util = "", ""
        ram = ""
        try:
            with open("/proc/meminfo") as f:
                for line in f:
                    if line.startswith("MemAvailable"):
                        ram = round(int(line.split()[1]) / 1048576, 2)
        except OSError:
            pass
        if ram != "":
            self.min_ram_gb = ram if self.min_ram_gb is None else min(self.min_ram_gb, ram)
        self.samples += 1
        with open(self.path, "a", encoding="utf-8") as f:
            f.write("%s,%s,%s,%s\n" % (datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M:%S"), gpu_mb, util, ram))

    def run(self):
        os.makedirs(os.path.dirname(self.path), exist_ok=True)
        with open(self.path, "w", encoding="utf-8") as f:
            f.write("time_utc,gpu_mem_used_mb,gpu_util_pct,ram_available_gb\n")
        while not self.stop_flag.is_set():
            try:
                self.sample()
            except Exception:
                pass
            self.stop_flag.wait(self.every)

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)
        return {"peak_gpu_mb": self.peak_gpu_mb, "peak_gpu_util_pct": self.peak_util, "min_ram_available_gb": self.min_ram_gb, "samples": self.samples}


def check_internet():
    import urllib.request
    bad_required = []
    for name, url, required in NET_HOSTS:
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "mica-benchmark-pilot"}), timeout=20) as r:
                r.read(512)
                log("  reachable: %-9s %s" % (name, r.status))
        except Exception as ex:
            log("  NOT reachable: %-9s %s" % (name, str(ex)[:100]))
            if required:
                bad_required.append(name)
    return bad_required


free_big, free_work = round(shutil.disk_usage(BIG).free / 1e9), round(shutil.disk_usage(WORK).free / 1e9)
cpu_model = sh("lscpu | grep -m1 'Model name'").split(":")[-1].strip()
log("platform %s | python %s | %s | %s CPU cores (%s) | %s GB RAM" % (PLATFORM, platform.python_version(), platform.platform(), os.cpu_count(), cpu_model, meminfo_gb()))
log("large files go to %s (%d GB free); results to %s (%d GB free)" % (BIG, free_big, WORK, free_work))
with open("machine.txt", "w", encoding="utf-8") as f:
    f.write(sh("nvidia-smi") + "\n\n" + sh("lscpu") + "\n\n" + sh("free -h") + "\n\n" + sh("df -h") + "\n\n" + sh("g++ --version | head -1; git --version; uname -a; cat /etc/os-release | head -3") + "\n")
STATE["gpu"] = sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") if shutil.which("nvidia-smi") else ""
if STATE["gpu"] and not STATE["gpu"].startswith("ERROR"):
    log("GPU: " + STATE["gpu"].replace("\n", " | "))
elif REQUIRE_GPU:
    fail("this session has no GPU. Kaggle: open Session options on the right, set Accelerator to GPU T4 x2 (or GPU P100), then Run all again. "
         "Colab: Runtime -> Change runtime type -> T4 GPU, then Run all again.")
if STATE["ok"]:
    log("checking the internet:")
    missing = check_internet() if NET_HOSTS else []
    if "PyPI" in missing or "GitHub" in missing:
        fail("the internet is off. Kaggle: open Session options on the right and switch Internet on (the account must be phone-verified), then Run all again.")
    elif missing:
        log("WARNING: %s cannot be reached from here; the step that needs it will tell you." % ", ".join(missing))


## Step 2. Write the scripts and the frozen data files

The files below are exact copies of the files in the repository (scripts, the AlphaFold Server job table and the hashes of the 26 predictions,
the entry list, the exact list of Python packages). The last cell of this step checks their hashes.

In [ ]:
import os
for d in ("scripts", "data/af3", "env", "logs", "gpu"):
    os.makedirs(d, exist_ok=True)
print("folders ready")


In [ ]:
%%writefile scripts/nbtools.py
#!/usr/bin/env python3
"""Tools that the Phase 4 notebooks share. They were added after the first MICA pilot, which showed three weaknesses:

1. a program that is silent for an hour (Phenix docking) looked like a frozen notebook: `Heartbeat` prints one line every few minutes
   (elapsed time, load, free RAM, the busiest processes, the last line of the program's own log);
2. the zip with the results was only written by the very last cell, so a stopped session left nothing: `pack` writes the zip in
   one step and the notebooks call it after every stage (a fixed name, replaced each time);
3. the last cell used variables of earlier cells and failed after the kernel was restarted: `recover` needs nothing from the notebook,
   it finds the work folder on disk and packs whatever is there.

Everything the zip should hold is copied into the work folder by the notebook, so that these functions only walk that folder.
Not packed: maps and other large data, files over 60 MB, the AlphaFold Server predictions of the user (they must stay private), and any file
that contains one of the secrets (the Phenix password).

    python scripts/nbtools.py --work <work folder> --out <folder for the zip> --notebook <name>      # the recovery path, by hand
"""
from __future__ import annotations

import argparse
import datetime
import json
import os
import subprocess
import sys
import threading
import time
import zipfile

MAX_FILE = 60 * 1000 * 1000
SKIP_DIRS = {"AF3_results", "af3_upload", "af3_unzipped", "grids", "AF3_encodings", "temp_maps", "native", "downloads", "scripts", "data", "env", "__pycache__"}
SKIP_SUFFIXES = (".map", ".mrc", ".ccp4", ".gz", ".part", ".tmp", ".zip")
WORK_CANDIDATES = ("/kaggle/working/pilot", "/content/pilot")


def utc_now():
    return datetime.datetime.now(datetime.timezone.utc)


def last_line(path, max_chars=160):
    """The last non-empty line of a text file, or "" if there is none or the file cannot be read."""
    try:
        with open(path, "rb") as f:
            f.seek(0, os.SEEK_END)
            size = f.tell()
            f.seek(max(0, size - 8192))
            tail = f.read().decode("utf-8", errors="replace")
    except OSError:
        return ""
    lines = [ln.strip() for ln in tail.splitlines() if ln.strip()]
    return lines[-1][:max_chars] if lines else ""


def machine_load():
    """(load average over 1 minute, free RAM in GB, text about the three busiest processes), each None or "" where it cannot be read."""
    load = free = None
    try:
        with open("/proc/loadavg") as f:
            load = float(f.read().split()[0])
    except (OSError, ValueError, IndexError):
        pass
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable"):
                    free = round(int(line.split()[1]) / 1048576, 1)
    except (OSError, ValueError, IndexError):
        pass
    top = ""
    try:
        r = subprocess.run(["ps", "-eo", "pcpu,etimes,comm", "--sort=-pcpu"], capture_output=True, text=True, timeout=10, errors="replace")
        rows = [ln.split(None, 2) for ln in r.stdout.splitlines()[1:4] if ln.strip()]
        top = ", ".join("%s %s%% (%ss)" % (c[2][:18], c[0], c[1]) for c in rows if len(c) == 3)
    except (OSError, subprocess.SubprocessError):
        pass
    return load, free, top


class Heartbeat(threading.Thread):
    """While a long program runs in the main thread, prints one line every `every` seconds through `log`. `watch` is a list of files whose last
    line is shown (the program's own log); `on_tick` is called after each line (the notebooks use it to refresh the partial zip)."""

    def __init__(self, label, log, watch=(), every=120, on_tick=None):
        super().__init__(daemon=True)
        self.label, self.log, self.watch, self.every, self.on_tick = label, log, list(watch), every, on_tick
        self.stop_flag, self.t0, self.ticks = threading.Event(), time.time(), 0

    def line(self):
        load, free, top = machine_load()
        minutes = (time.time() - self.t0) / 60
        parts = ["[%s] running for %.0f min" % (self.label, minutes) if minutes >= 1 else "[%s] running for %.0f s" % (self.label, time.time() - self.t0)]
        if load is not None:
            parts.append("load %.1f" % load)
        if free is not None:
            parts.append("free RAM %.1f GB" % free)
        if top:
            parts.append("busiest: " + top)
        for path in self.watch:
            if os.path.exists(path):
                age = time.time() - os.path.getmtime(path)
                parts.append("%s: %d bytes, changed %s ago, last line: %s" % (os.path.basename(path), os.path.getsize(path),
                                                                           "%.0f s" % age if age < 120 else "%.0f min" % (age / 60), last_line(path) or "(empty)"))
        return " | ".join(parts)

    def run(self):
        while not self.stop_flag.wait(self.every):
            try:
                self.log(self.line())
                self.ticks += 1
                if self.on_tick:
                    self.on_tick()
            except Exception as ex:                       # a failing heartbeat must never stop the program it watches
                try:
                    self.log("[%s] heartbeat problem: %s" % (self.label, str(ex)[:150]))
                except Exception:
                    pass

    def stop(self):
        self.stop_flag.set()
        self.join(timeout=30)


def collect(work, skip_dirs=SKIP_DIRS, skip_suffixes=SKIP_SUFFIXES):
    """All files below `work` that may go into the zip, in a fixed order."""
    files = []
    for dirpath, dirs, names in os.walk(work):
        dirs[:] = sorted(d for d in dirs if d not in skip_dirs)
        for n in sorted(names):
            if not n.endswith(skip_suffixes):
                files.append(os.path.join(dirpath, n))
    return files


def pack(work, zip_path, root_name, secrets=(), max_file=MAX_FILE, extra=()):
    """Writes `zip_path` in one step (a temporary file that replaces the old zip at the end, so a stop in the middle never leaves a broken zip).
    Returns (number of files, list of what was left out and why). `extra` are files outside `work` that go in as well."""
    wanted = collect(work) + [p for p in extra if os.path.isfile(p)]
    left_out, tmp = [], zip_path + ".tmp"
    try:
        with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
            for path in dict.fromkeys(wanted):
                try:
                    if os.path.getsize(path) > max_file:
                        left_out.append("%s (larger than %d MB)" % (os.path.relpath(path, work), max_file // 1000000))
                        continue
                    with open(path, "rb") as f:
                        data = f.read()
                except OSError as ex:
                    left_out.append("%s (%s)" % (os.path.relpath(path, work), str(ex)[:60]))
                    continue
                if any(s and s.encode() in data for s in secrets):
                    left_out.append("%s (contained the password)" % os.path.relpath(path, work))
                    continue
                inside = os.path.relpath(path, work) if path.startswith(work + os.sep) else os.path.join("outside", path.lstrip(os.sep))
                z.writestr(root_name + "/" + inside, data)
            n = len(z.namelist())
        os.replace(tmp, zip_path)
    except BaseException:
        try:
            os.remove(tmp)
        except OSError:
            pass
        raise
    return n, left_out


def partial_zip_path(out_dir, notebook):
    return os.path.join(out_dir, "%s_partial.zip" % notebook)


def final_pack(work, out_dir, notebook, complete, secrets=(), extra=(), note=""):
    """The zip that is sent to me. complete: True (everything ran), False (something stopped) or None (unknown: recovered after a restart).
    The partial zip is removed once this one exists."""
    stamp = utc_now().strftime("%Y%m%d-%H%M")
    suffix = "" if complete else ("_INCOMPLETE" if complete is False else "_RECOVERED")
    name = "%s_result_%s%s" % (notebook, stamp, suffix)
    path = os.path.join(out_dir, name + ".zip")
    n, left_out = pack(work, path, name, secrets=secrets, extra=extra)
    try:
        os.remove(partial_zip_path(out_dir, notebook))
    except OSError:
        pass
    return path, n, left_out


def find_work(env=None):
    """The work folder of a notebook that ran in this session: the one with a run.log."""
    env = os.environ if env is None else env
    for cand in (env.get("MICA_WORK"),) + WORK_CANDIDATES + (os.path.abspath("pilot"),):
        if cand and os.path.isfile(os.path.join(cand, "run.log")):
            return cand
    return None


def known_secrets(env=None):
    """The Phenix password, if it can be read again (environment, Kaggle or Colab secrets), to check the files for it after a restart."""
    env = os.environ if env is None else env
    found = []
    if env.get("PHENIX_PASSWORD"):
        found.append(env["PHENIX_PASSWORD"].strip())
    for getter in ("kaggle", "colab"):
        try:
            if getter == "kaggle":
                from kaggle_secrets import UserSecretsClient
                v = UserSecretsClient().get_secret("PHENIX_PASSWORD")
            else:
                from google.colab import userdata
                v = userdata.get("PHENIX_PASSWORD")
            if v:
                found.append(v.strip())
        except Exception:
            pass
    return [s for s in dict.fromkeys(found) if len(s) >= 6]


def read_state(work):
    try:
        with open(os.path.join(work, "state.json"), encoding="utf-8") as f:
            return json.load(f)
    except (OSError, ValueError):
        return {}


def recover(notebook, out_dir=None, env=None, say=print):
    """The last step when nothing from the notebook is left in memory (kernel restarted, cell run alone, ...). Returns the zip path or None."""
    env = os.environ if env is None else env
    work = find_work(env)
    if not work:
        say("RECOVERY: no work folder with a run.log was found (looked in %s). Nothing to pack." % ", ".join(WORK_CANDIDATES))
        return None
    out_dir = out_dir or env.get("MICA_WORK") or os.path.dirname(work)
    state = read_state(work)
    info = {"notebook": notebook, "recovered": True, "recovered_utc": utc_now().isoformat(), "state_json_found": bool(state), "state": state}
    with open(os.path.join(work, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    path, n, left_out = final_pack(work, out_dir, notebook, complete=None, secrets=known_secrets(env))
    say("RECOVERY: zip written from the files that are on disk: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        say("left out of the zip: %s" % "; ".join(left_out))
    say("If a notebook cell stopped with an error, send me this zip and the text of the error.")
    return path


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--work", required=True)
    p.add_argument("--out", required=True)
    p.add_argument("--notebook", required=True)
    args = p.parse_args(argv)
    path, n, left_out = final_pack(args.work, args.out, args.notebook, complete=None)
    print("%s (%d files)" % (path, n))
    for x in left_out:
        print("left out:", x)
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/mica_inputs.py
#!/usr/bin/env python3
"""Phase 4: the input folder that MICA expects, for one entry of the frozen list.

MICA reads (README, "Running on New Dataset", and utils/process_AF3_results.py):

    <root>/<PDB ID>/
        <pdb id>.fasta                         one record per unique chain sequence, header  >26DK_1|Chains A, B|...
        emd_<n>.map                            (put there by the notebook)
        AF3_results/<pdb id>_<k>/fold_<pdb id>_<k>_model_0.cif

The header must start with the PDB ID, one underscore, the number of the sequence, then "|Chains ..." (the number of chains is
the number of copies of that sequence in the entry). The third part of the file name (split at "_") must be the number k, and
the PDB ID of the first header must be the one of the files, or MICA finds no protein for the file and drops it silently.
AlphaFold Server returned the files as fold_<job name in lower case>_model_0.cif with the job names of data/af3/af3_jobs.csv
(for example fold_26dk_e1_model_0.cif); this script renames E1, E2, ... to 1, 2, ... and writes the FASTA file from the same table, so the
sequence MICA reads is the sequence that AlphaFold Server was given (letters outside the 20 amino acids were replaced by G there).

    python scripts/mica_inputs.py --jobs data/af3/af3_jobs.csv --models <folder with the 26 files> --pdb 26DK --out <root>
"""
from __future__ import annotations

import argparse
import csv
import json
import os
import shutil
import sys

SCRIPT_VERSION = "1.0"
CHAIN_LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"


def read_jobs(path, pdb_id):
    """The rows of af3_jobs.csv for one entry, in the order of the entities (E1, E2, ...)."""
    with open(path, newline="", encoding="utf-8") as f:
        rows = [r for r in csv.DictReader(f) if r["pdb_id"].upper() == pdb_id.upper()]
    if not rows:
        raise ValueError("%s: no job for %s" % (path, pdb_id))
    rows.sort(key=lambda r: int(r["entity"].lstrip("Ee")))
    numbers = [int(r["entity"].lstrip("Ee")) for r in rows]
    if numbers != list(range(1, len(rows) + 1)):
        raise ValueError("%s: the entities of %s are %s, expected 1 to %d without gaps" % (path, pdb_id, numbers, len(rows)))
    return rows


def find_model(models_dir, job_name):
    """Path of fold_<job name in lower case>_model_0.cif somewhere under models_dir (case does not matter)."""
    want = "fold_%s_model_0.cif" % job_name.lower()
    hits = []
    for root, _dirs, files in os.walk(models_dir):
        hits += [os.path.join(root, f) for f in files if f.lower() == want]
    if not hits:
        raise ValueError("no file %s under %s" % (want, models_dir))
    if len(hits) > 1:
        raise ValueError("%s occurs %d times under %s" % (want, len(hits), models_dir))
    return hits[0]


def mica_fasta(rows):
    """FASTA text with MICA's header format. The chain letters run through the entry (A, B for the first sequence, C, D for the next, ...)."""
    total = sum(int(r["copies_in_entry"]) for r in rows)
    if total > len(CHAIN_LETTERS):
        raise ValueError("%s has %d chains; MICA writes PDB files and cannot name more than %d" % (rows[0]["pdb_id"], total, len(CHAIN_LETTERS)))
    out, used = [], 0
    for k, r in enumerate(rows, 1):
        copies = int(r["copies_in_entry"])
        if copies < 1:
            raise ValueError("%s: copies_in_entry is %s" % (r["job_name"], r["copies_in_entry"]))
        letters = ", ".join(CHAIN_LETTERS[used:used + copies])
        used += copies
        pdb = r["pdb_id"].upper()
        if "_" in pdb or "|" in pdb:
            raise ValueError("the PDB ID %r contains _ or |" % pdb)
        out.append(">%s_%d|Chains %s|%s sequence %d|" % (pdb, k, letters, pdb, k))
        out.append(r["sequence_submitted"])
    return "\n".join(out) + "\n"


def build(jobs_csv, models_dir, pdb_id, out_root):
    """Writes <out_root>/<PDB ID>/ with the FASTA file and AF3_results; returns a dict that says what was written."""
    rows = read_jobs(jobs_csv, pdb_id)
    pdb = rows[0]["pdb_id"].upper()
    low = pdb.lower()
    folder = os.path.abspath(os.path.join(out_root, pdb))
    results = os.path.join(folder, "AF3_results")
    # MICA derives its other folders with AF3_results_path.replace("results", ...): the word must occur only once, in the last part
    if "results" in os.path.dirname(results):
        raise ValueError("the path %s contains the word 'results' before AF3_results; MICA would write into the wrong folders. Use another root folder" % folder)
    os.makedirs(results, exist_ok=True)
    fasta_path = os.path.join(folder, low + ".fasta")
    with open(fasta_path, "w", encoding="utf-8") as f:
        f.write(mica_fasta(rows))
    files = []
    for k, r in enumerate(rows, 1):
        src = find_model(models_dir, r["job_name"])
        d = os.path.join(results, "%s_%d" % (low, k))
        os.makedirs(d, exist_ok=True)
        dst = os.path.join(d, "fold_%s_%d_model_0.cif" % (low, k))
        shutil.copyfile(src, dst)
        files.append({"job_name": r["job_name"], "source": os.path.basename(src), "target": os.path.relpath(dst, folder)})
    return {"pdb_id": pdb, "folder": folder, "fasta": fasta_path, "af3_results": results, "files": files}


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--jobs", required=True, help="data/af3/af3_jobs.csv")
    p.add_argument("--models", required=True, help="folder with the fold_*_model_0.cif files from AlphaFold Server")
    p.add_argument("--pdb", required=True, help="PDB ID of the entry, for example 26DK")
    p.add_argument("--out", required=True, help="root folder; the entry gets <out>/<PDB ID>/")
    args = p.parse_args(argv)
    try:
        info = build(args.jobs, args.models, args.pdb, args.out)
    except (ValueError, OSError, KeyError) as e:
        print("ERROR: %s" % e, file=sys.stderr)
        return 2
    print(json.dumps(info, indent=1))
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile scripts/evaluate_models.py
#!/usr/bin/env python3
"""Phase 4: scores one predicted model against the deposited model (docs/SPEC.md, decision 7), version 0.1 for the pilot.

What it computes with US-align (protein chains only, `-mol protein`, tabular output `-outfmt 2`; the prediction is structure 1 and the
deposited model is structure 2, so TM2 is the TM-score normalised by the length of the deposited model, the one that the US-align
output recommends):

    tm_score            TM2, normalised by the deposited model
    tm_score_pred_norm  TM1, normalised by the prediction (for reference only)
    ca_rmsd             RMSD over the aligned residues
    aligned_ca_length   number of aligned residues (Lali)
    seq_identity        identical residues among the aligned ones (IDali)
    pred_residues       protein residues in the prediction
    native_residues     protein residues in the deposited model (US-align L2)
    coverage            built residues / deposited residues; the deposited count is `modeled_residues` of data/entries_2026.csv

The Phenix part (Calpha match and sequence match from phenix.chain_comparison, and with them the Calpha quality score) is not here yet:
the pilot saves the raw Phenix output so that the exact command and the output lines can be fixed before the scripts are trusted.

    python scripts/evaluate_models.py --usalign <path to USalign> --pred model.pdb --native 26DK.cif --modeled-residues 191
"""
from __future__ import annotations

import argparse
import json
import subprocess
import sys

SCRIPT_VERSION = "0.1"
TABLE_COLUMNS = ["PDBchain1", "PDBchain2", "TM1", "TM2", "RMSD", "ID1", "ID2", "IDali", "L1", "L2", "Lali"]


def parse_usalign_tabular(text):
    """The rows of `USalign -outfmt 2`: a list of dicts with numbers as floats (L1, L2 and Lali as integers)."""
    rows = []
    for line in text.splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) != len(TABLE_COLUMNS):
            raise ValueError("a US-align line has %d fields, %d expected: %r" % (len(parts), len(TABLE_COLUMNS), line[:200]))
        row = dict(zip(TABLE_COLUMNS, parts))
        for k in ("TM1", "TM2", "RMSD", "ID1", "ID2", "IDali"):
            row[k] = float(row[k])
        for k in ("L1", "L2", "Lali"):
            row[k] = int(row[k])
        rows.append(row)
    return rows


def run_usalign(usalign, pred, native, timeout=900):
    """US-align of the prediction onto the deposited model; returns the one table row, or raises RuntimeError with the reason."""
    cmd = [usalign, pred, native, "-mol", "protein", "-outfmt", "2"]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, errors="replace")
    except subprocess.TimeoutExpired:
        raise RuntimeError("US-align did not finish within %d seconds" % timeout)
    except OSError as e:
        raise RuntimeError("US-align could not be started: %s" % e)
    if r.returncode != 0:
        raise RuntimeError("US-align stopped with exit code %d: %s" % (r.returncode, (r.stderr or r.stdout).strip()[-300:]))
    rows = parse_usalign_tabular(r.stdout)
    if len(rows) != 1:
        raise RuntimeError("US-align gave %d result lines, 1 expected. Output: %s" % (len(rows), r.stdout.strip()[-300:]))
    return rows[0]


def evaluate(usalign, pred, native, modeled_residues):
    """The numbers of one model. status is "ok", or the reason why no alignment exists (then the numbers are empty)."""
    out = {"status": "ok", "note": "", "tm_score": "", "tm_score_pred_norm": "", "ca_rmsd": "", "aligned_ca_length": "", "seq_identity": "",
           "pred_residues": "", "native_residues": "", "coverage": ""}
    try:
        row = run_usalign(usalign, pred, native)
    except (RuntimeError, ValueError) as e:
        out.update(status="no_alignment", note=str(e)[:300])
        return out
    out.update(tm_score=row["TM2"], tm_score_pred_norm=row["TM1"], ca_rmsd=row["RMSD"], aligned_ca_length=row["Lali"], seq_identity=row["IDali"],
               pred_residues=row["L1"], native_residues=row["L2"])
    if modeled_residues:
        out["coverage"] = round(row["L1"] / float(modeled_residues), 4)
    return out


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--usalign", required=True, help="path to the USalign program")
    p.add_argument("--pred", required=True, help="predicted model (pdb or cif)")
    p.add_argument("--native", required=True, help="deposited model (pdb or cif)")
    p.add_argument("--modeled-residues", type=int, default=0, help="modeled_residues of the entry in data/entries_2026.csv")
    p.add_argument("--out", help="write the numbers as json")
    args = p.parse_args(argv)
    res = evaluate(args.usalign, args.pred, args.native, args.modeled_residues)
    print(json.dumps(res, indent=1))
    if args.out:
        with open(args.out, "w", encoding="utf-8") as f:
            json.dump(res, f, indent=1)
    return 0 if res["status"] == "ok" else 2


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile data/af3/af3_jobs.csv
job_no,job_name,pdb_id,entity,stratum,seed_rank,length,copies_in_entry,sequence_submitted,substitution
1,9T5K_E1,9T5K,E1,le3.0,1,453,1,MHHHHHHGKPIPNPLLGLDSTENLYFQGIDPFTMFSTLKNETEKFREYQANLGRQGKPLASTATLTTKIIVYNPSNKSPALRWEITKFAMRLIWSPAASHSVKVGAALTLLSAHAENPGAMIRSLVNDPDIEVVITDISEFDHGVPRLERRGEKAEQQMDSYRRILDRAPQENLFYNPEVDDLEILDSGTFLFAIATVLAQVWILVAKAVTAPDTAEESENKRWAKYVQQKRVNPDYLVSNRWITAMRSLISIDLSVRKYMVEILIEVKKSGVARGRLNEMIADIGNYIEETGMAGFFLTIKYGLEMKFPVIVINEFQADLLTLQTLMRTYMDLGPRAPYMVLLEDSIQTKFAPGNYPLLWSFAMGVGTTLDRSMGALNINRSYLEPIYFKLGQNAARKNAGSIDRKLAEELGLTQEQANEIKEMMQEVTTQRHETNVQAREGKFNVAAGGIE,
2,22FX_E1,22FX,E1,le3.0,2,182,1,MTTASPSQVRQNYHQDAEAAINRQINLELYASYVYLSMSCYFDRDDVALKNFAKYFLHQSHEEREHAEKLMKLQNQRGGRIFLQDIKKPDRDDWESGLNAMECALHLEKSVNQSLLELHKLATDKNDPHLCDFIETYYLSEQVKSIKELGDHVTNLRKMGAPEAGMAEYLFDKHTLGHGDES,
3,9JKK_E1,9JKK,E1,le3.0,3,620,1,MSKSKCSVGLMSSVVAPAKEPNAVGPKEVELILVKEQNGVQLTSSTLTNPRQSPVEAQDRETWGKKIDFLLSVIGFAVDLANVWRFPYLCYKNGGGAFLVPYLLFMVIAGMPLFYMELALGQFNREGAAGVWKICPILKGVGFTVILISLYVGFFYNVIIAWALHYLFSSFTTELPWIHCNNSWNSPNCSDAHPGDSSGDSSGLNDTFGTTPAAEYFERGVLHLHQSHGIDDLGPPRWQLTACLVLVIVLLYFSLWKGVKTSGKVVWITATMPYVVLTALLLRGVTLPGAIDGIRAYLSVDFYRLCEASVWIDAATQVCFSLGVGFGVLIAFSSYNKFTNNCYRDAIVTTSINSLTSFSSGFVVFSFLGYMAQKHSVPIGDVAKDGPGLIFIIYPEAIATLPLSSAWAVVFFIMLLTLGIDSAMGGMESVITGLIDEFQLLHRHRELFTLFIVLATFLLSLFCVTNGGIYVFTLLDHFAAGTSILFGVLIEAIGVAWFYGVGQFSDDIQQMTGQRPSLYWRLCWKLVSPCFLLFVVVVSIVTFRPPHYGAYIFPDWANALGWVIATSSMAMVPIYAAYKFCSLPGSFREKLAYAIAPEKDRELVDRGEVRQFTLRHWLKV,
4,9TKA_E1,9TKA,E1,le3.0,4,293,1,MNPASDGGTSESIFDLDYASWGIRSTLMVAGFVFYLGVFVVCHQLSSSLNATYRSLVAREKVFWDLAATRAVFGVQSTAAGLWALLGDPVLHADKARGQQNWCWFHITTATGFFCFENVAVHLSNLIFRTFDLFLVIHHLFAFLGFLGCLVNLQAGHYLAMTTLLLEMSTPFTCVSWMLLKAGWSESLFWKLNQWLMIHMFHCRMVLTYHMWWVCFWHWDGLVSSLYLPHLTLFLVGLALLTLIINPYWTHKKTQQLLNPVDWNFAQPEAKSRPEGNGQLLRKKRPDAAIEGR,
5,9WQ6_E1,9WQ6,E1,gt3.0_le4.0,4,339,1,LSLLHLQEKNWSALLTAVVIILTIAGNILVIMAVSLEKKLQNATNYFLMSLAIADMLLGFLVMPVSMLTILYGYRWPLPSKLCAVWIYLDVLFSTASIMHLCAISLDRYVAIQNPIHHSRFNSRTKAFLKIIAVWTISVGISMPIPVFGLQDDSKVFKEGSCLLADDNFVLIGSFVSFFIPLTIMVITYFLTIKSLQKEATLCVSDLGTRAKLASFSFLPQSSLSSEKLFQRSIHREPGSYTGRRTMQSISNEQKACKVLGIVFFLFVVMWCPFFITNIMAVICKESCNEDVIGALLNVFVWIGYLSSAVNPLVYTLFNKTYRSAFSRYIQCQYKENKK,
6,9WFL_E1,9WFL,E1,le3.0,5,697,1,SRIGKLLGFEWTDLSSWRRLVTLLNRPTDPASLAVFRFLFGFLMVLDIPQERGLSSLDRKYLDGLDVCRFPLLDALRPLPLDWMYLVYTIMFLGALGMMLGLCYRISCVLFLLPYWYVFLLDKTSWNNHSYLYGLLAFQLTFMDANHYWSVDGLLNAHRRNAHVPLWNYAVLRGQIFIVYFIAGVKKLDADWVEGYSMEYLSRHWLFSPFKLLLSEELTSLLVVHWGGLLLDLSAGFLLFFDVSRSIGLFFVSYFHCMNSQLFSIGMFSYVMLASSPLFCSPEWPRKLVSYCPRRLQQLLPLKAAPQPSVSCVYKRSRGKSGQKPGLRHQLGAAFTLLYLLEQLFLPYSHFLTQGYNNWTNGLYGYSWDMMVHSRSHQHVKITYRDGRTGELGYLNPGVFTQSRRWKDHADMLKQYATCLSRLLPKYNVTEPQIYFDIWVSINDRFQQRIFDPRVDIVQAAWSPFQRTSWVQPLLMDLSPWRAKLQEIKSSLDNHTEVVFIADFPGLHLENFVSEDLGNTSIQLLQGEVTVELVAEQKNQTLREGEKMQLPAGEYHKVYTTSPSPSCYMYVYVNTTELALEQDLAYLQELKEKVENGSETGPLPPELQPLLEGEVKGGPEPTPLVQTFLRRQQRLQEIERRRNTPFHERFFRFLLRKLYVFRRSFLMTCISLRNLILGRPSLEQLAQEVTYANLRPF,
7,9WFL_E2,9WFL,E2,le3.0,5,29,1,RVFLTGEKANSILKRYPRANGFFEEIRQG,
8,9V4D_E1,9V4D,E1,gt3.0_le4.0,5,393,1,MGKTIIALSYIFCLVFADYKDDDDAANFTPVNGSSGNQSVRLVTSSSLEVLFQGPGSGNDSVSYEYGDYSDLSDRPVDCLDGACLAIDPLRVAPLPLYAAIFLVGVPGNAMVAWVAGKVARRRVGATWLLHLAVADLLCCLSLPILAVPIARGGHWPYGAVGCRALPSIILLTMYASVLLLAALSADLCFLALGPAWWSTVQRACGVQVACGAAWTLALLLTVPSAIYRRLHQEHFPARLQCVVDYGGSSSTENAVTAIRFLFGFLGPLVAVASCHSALLCWAARRCRPLGTAIVVGFFVCWAPYHLLGLVLTVAAPNSALLARALRAEPLIVGLALAHSCLNPMLFLYFGRAQLRRSLPAACHWALRESQGQDESVDSKKSTSHDLVSEMEV,
9,9V4D_E2,9V4D,E2,gt3.0_le4.0,5,9,1,GYKPLGLAY,1 residue(s) X replaced by G
10,9VI4_E1,9VI4,E1,le3.0,6,516,1,VGNASGNWHCDSTWLGDRVITTSTRTWALPTYNNHLYKQISSASTGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTTNDGVTTIANNLTSTVQVFSDSEYQLPYVLGSAHQGCLPPFPADVFMIPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLNRTQNQSGSAQNKDLLFSRGSPAGMSVQPKNWLPGPCYRQQRVSKTKTDNNNSNFTWTGASKYNLNGRESIINPGTAMASHKDDKDKFFPMSGVMIFGKESAGASNTALDNVMITDEEEIKATNPVATERFGTVATNHQSVNYEEETGDVHVMGALPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPPAEFSATKFASFITQYSTGQVSVEIEWELQKENSKRWNPEVQYTSNYAKSANVDFTVDNNGLYTEPRPIGTRYLTRPL,
11,9ZLH_E1,9ZLH,E1,le3.0,7,206,1,DFVPCPVDTVQIVLPPHLERIREKLAENIHELWALTRIEQGWTYGPVRDDNKRLHPCLVNFHSLPEPERNYNLQMSGETLKTLLALGCHVGMADEKAEDNLKKTKLPKTYMMSNGYKPAPLDLSHVRLTPAQTTLVDRLAENGHNVWARDRVAQGWSYSAVQDIPARRNPRLVPYRLLDEATKRSNRDSLCQAVRTLLGYGYNIEP,
12,9OTA_E1,9OTA,E1,gt3.0_le4.0,7,270,2,GALLAGALAAYAAYLVLGALLVARLEGPHEARLRAELETLRAQLLQRSPCVAAPALDAFVERVLAAGRLGRVVLANASGSANASDPAWDFASALFFASTLITTVGYGYTTPLTDAGKAFSIAFALLGVPTTMLLLTASAQRLSLLLTHVPLSWLSMRWGWDPRRAACWHLVALLGVVVTVCFLVPAVIFAHLEEAWSFLDAFYFCFISLSTIGLGDYVPGEAPGQPYRALYKVLVTVYLFLGLVAMVLVLQTFRHVSDLHGLTELILLPP,
13,9VUH_E1,9VUH,E1,le3.0,8,332,1,MADYKDDDDKSGRGAMGALQGCRSVEEFQCLNRIEEGTYGVVYRAKDKKTDEIVALKRLKMEKEKEGFPITSLREINTILKAQHPNIVTVREIVVGSNMDKIYIVMNYVEHDLKSLMETMKQPFLPGEVKTLMIQLLRGVKHLHDNWILHRDLKTSNLLLSHAGILKVGDFGLAREYGSPLKAYTPVVVTLWYRAPELLLGAKEYSTAVDMWSVGCIFGELLTQKPLFPGKSEIDQINKVFKDLGTPSEKIWPGYSELPAVKKMTFSEHPYNNLRKRFGALLSDQGFDLMNKFLTYFPGRRISAEDGLKHEYFRETPLPIDPSMFPKLVEKY,
14,9VUH_E2,9VUH,E2,le3.0,8,365,1,MASGPHSTATAAAAASSAAPSAGGSSSGTTTTTTTTTGGILIGDRLYSEVSLTIDHSLIPEERLSPTPSMQDGLDLPSETDLRILGCELIQAAGILLRLPQVAMATGQVLFHRFFYSKSFVKHSFEIVAMACINLASKIEEAPRRIRDVINVFHHLRQLRGKRTPSPLILDQNYINTKNQVIKAERRVLKELGFCVHVKHPHKIIVMYLQVLECERNQTLVQTAWNYMNDSLRTNVFVRFQPETIACACIYLAARALQIPLPTRPHWFLLFGTTEEEIQEICIETLRLYTRKKPNYELLEKEVEKRKVALQEAKLKAKGLNPDGRIPGLIWSHPQFEKGGGSGGGGGSGGGSGGSSAWSHPQFEK,
15,9VUH_E3,9VUH,E3,le3.0,8,318,1,MGSENLYFQGMAGKKNVLSSLAVYAEDSEPESDGEAGIEAVGSTAEEKGGLVSDAYGEDDFSRLGGDEDGYEEEEDENSRQSEDDDSETEKPEADDPKDNTEAEKRDPQELVASFSERVRNMSPDEIKIPPEPPGRCSNHLQDKIQKLYERKIKEGMDMNYIIQRKKEFRNPSIYEKLIQFCAIDELGTNYPKDMFDPHGWSEDSYYEALAKAQKIEMDKLEKAKKERTKIEFVTGTKKGTTTNATSTTTTTASTAVADAQKRKSKWDSAIPVTTIAQPTILTTTATLPAVVTVTTSASGSKTTVISAVGTIVKNAKQ,
16,9KQM_E1,9KQM,E1,gt3.0_le4.0,8,637,1,MADLEDNWETLNDNLKVIEKADNAAQVKDALTKMRAAALDAQKATPPKLEDKSPDSPEMKDFRHGFDILVGQIDDALKLANEGKVKEAQAAAEQLKTTRNAYIQKYLLQEPRRQRRLVLVIVCVALLLDNMLYMVIVPIVPDYIAHMRGGGEGPTRTPEVWEPTLPLPTPANASAYTANTSASPTAAWPAGSALRPRYPTESEDVKIGVLFASKAILQLLVNPLSGPFIDRMSYDVPLLIGLGVMFASTVLFAFAEDYATLFAARSLQGLGSAFADTSGIAMIADKYPEEPERSRALGVALAFISFGSLVAPPFGGILYEFAGKRVPFLVLAAVSLFDALLLLAVAKPFSAAARARANLPVGTPIHRLMLDPYIAVVAGALTTCNIPLAFLEPTIATWMKHTMAASEWEMGMAWLPAFVPHVLGVYLTVRLAARYPHLQWLYGALGLAVIGASSCIVPACRSFAPLVVSLCGLCFGIALVDTALLPTLAFLVDVRHVSVYGSVYAIADISYSVAYALGPIVAGHIVHSLGFEQLSLGMGLANLLYAPVLLLLRNVGLLTRSRSERDVLLDEPPQGLYDAVRLRERPVSGQDGEPRSPPGPFDACEDDYNYYYTRSHHHHHHHHHHGSVEDYKDDDDK,
17,9WME_E1,9WME,E1,gt3.0_le4.0,9,453,1,MTPSISKDAPIKGSITISKKGATFTAYKLLDAIKSGDAYEYSVNSDLKDFFNNSNYGSYSQESIQKLNGEQVKEFAINLHKYILENKKSGQELKDGQKNTVDLGYYLVTETSSDSEGAAVASTPIIVSVPQVSGDSWNYDVTINPKDNTPILEKNIVKENQRVKTSSENIGDVVKYEVKASIPVYQKNAQNIMYKFTDTMSKGLTYDEKTGFKVTSGDKVFAKDTDYTVDVKKQEDGSTVITINFVYENIKAYAETGITLNYQATLNKDAVISNKENLGNTNNIQLDYTNNPHVKDSYKKLTDKVTTYTFGFGITKVDSELNSKLLQGAEFSVKDAGGKIVAKYTYDEKGQVVYLSGNGVTNSKGITTFLGLKEGKYFITEEVAPSGYSLLKNPVEVTITANKDESGNYTGAATIEISNGNKAGQIINDISEKDGNILFNVQIENHAHHHHHH,
18,9WME_E2,9WME,E2,gt3.0_le4.0,9,657,1,MEETKSVQNDGAVEITSTTFESNTVAKGISNNLRIDYKILNKDKLKDGDKIVISLPDIFKDIEPKCHDQHFKDFDVKDGVVTLTFNENVEKAVTGYMIIRFVGNSNIRKGVSYPVSIDLNGKPSTVYITGEEYSNSSSGAQYPLMYKTADLPTAATDKEQGREYYGEIVDRNKPIKYFVEINLGDGVNPNTRSYLSNADFFDNIPKGMALDVNSICIKRMGYYDERSSDVTKDFWESNRIKADTKHLEINFGDIRYERYTVIYETRITSTESGYLNDAKLYYDDKELPSKHYSKLSKDAGALNVYKYVDKTKVKNNLNDQKIKYDIKFDSYGYFFKDTLNIIDKLDPRLSDIKITATDQFITDFDENTKELVIKNSNGDIDAKKPAYITIEASMKNVGPGEVVKNIAYVNGNPTNEVSTRKNPIVEIIKVSKDDVESNLLEGAIFKLTTKDGKTVKDVYNKGVKTFTTSSEGSIRFELPNGDYKLEEIKAPNGYKLDQTPIEFTVNDESKIVKVVAKDDPLPTTCNLVINKINEKEIPILGAKFKLFEESNPKKVLKFSSQKNNYELNQNGVGKLTPSGKNASFKINNLHYGNYILKEVQAPKGYKLSDDIYITLGFEESFYRVGKQGEKIILNKNTETNTYNISVENVPRIILPET,
19,9OC0_E1,9OC0,E1,le3.0,11,847,2,MSTMHLLTFALLFSCSFARAASDPKIVNIGAVLSTRKHEQMFREAVNQANKRHGSWKIQLQATSVTHKPNAIQMALSVCEDLISSQVYAILVSHPPTPNDHFTPTPVSYTAGFYRIPVLGLTTRMSIYSDKSIHLSFLRTVPPYSHQSSVWFEMMRVYNWNHIILLVSDDHEGRAAQKRLETLLEERESKAEKVLQFDPGTKNVTALLMEARELEARVIILSASEDDAATVYRAAAMLDMTGSGYVWLVGEREISGNALRYAPDGIIGLQLINGKNESAHISDAVGVVAQAVHELLEKENITDPPRGCVGNTNIWKTGPLFKRVLMSSKYADGVTGRVEFNEDGDRKFAQYSIMNLQNRKLVQVGIYNGTHVIPNDRKIIWPGGETEKPRGYQMSTRLKIVTIHQEPFVYVKPTMSDGTCKEEFTVNGDPVKKVICTGPNDTSPGSPRHTVPQCCYGFCIDLLIKLARTMQFTYEVHLVADGKFGTQERVQNSNKKEWNGMMGELLSGQADMIVAPLTINNERAQYIEFSKPFKYQGLTILVKKEIPRSTLDSFMQPFQSTLWLLVGLSVHVVAVMLYLLDRFSPFGRFKVNSEEEEEDALTLSSAMWFSWGVLLNSGIGEGAPRSFSARILGMVWAGFAMIIVASYTANLAAFLVLDRPEERITGINDPRLRNPSDKFIYATVKQSSVDIYFRRQVELSTMYRHMEKHNYESAAEAIQAVRDNKLHAFIWDSAVLEFEASQKCDLVTTGELFFRSGFGIGMRKDSPWKQQVSLSILKSHENGFMEDLDKTWVRYQECDSRSNAPATLTFENMAGVFMLVAGGIVAGIFLIFIEIAYKRHKDANGAQ,
20,9OC0_E2,9OC0,E2,le3.0,11,862,2,GWSHPQFEKGGGSGGGSGGSAWSHPQFEKGALVPRGRSQKSPPSIGIAVILVGTSDEVAIKDAHEKDDFHHLSVVPRVELVAMNETDPKSIITRICDLMSDRKIQGVVFADDTDQEAIAQILDFISAQTLTPILGIHGGSSMIMADKDESSMFFQFGPSIEQQASVMLNIMEEYDWYIFSIVTTYFPGYQDFVNKIRSTIENSFVGWELEEVLLLDMSLDDGDSKIQNQLKKLQSPIILLYCTKEEATYIFEVANSVGLTGYGYTWIVPSLVAGDTDTVPSEFPTGLISVSYDEWDYGLPARVRDGIAIITTAASDMLSEHSFIPEPKSSCYNTHEKRIYQSNMLNRYLINVTFEGRNLSFSEDGYQMHPKLVIILLNKERKWERVGKWKDKSLQMKYYVWPRMCPETEEQEDDHLSIVTLEEAPFVIVESVDPLSGTCMRNTVPCQKRIISENKTDEEPGYIKKCCKGFCIDILKKISKSVKFTYDLYLVTNGKHGKKINGTWNGMIGEVVMKRAYMAVGSLTINEERSEVVDFSVPFIETGISVMVSRSNGTVSPSAFLEPFSADVWVMMFVMLLIVSAVAVFVFEYFSPVGYNRCLADGREPGGPSFTIGKAIWLLWGLVFNNSVPVQNPKGTTSKIMVSVWAFFAVIFLASYTANLAAFMIQEEYVDQVSGLSDKKFQRPNDFSPPFRFGTVPNGSTERNIRNNYAEMHAYMGKFNQRGVDDALLSLKTGKLDAFIYDAAVLNYMAGRDEGCKLVTIGSGKVFASTGYGIAIQKDSGWKRQVDLAILQLFGDGEMEELEALWLTGICHNEKNEVMSSQLDIDNMAGVFYMLGAAMALSLITFICEHLFYWQFRHSFMG,
21,13BD_E1,13BD,E1,le3.0,12,1271,2,MFVFLVLLLLVSSQCVNLTTTTQLPLAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFSNVTWFHVISGTNGTKRFDNPVLPFNDGVYFASIEKSNITRGWIFGTTLDSKTQSLLIVNNATNVVIKVCEFQFNNKSWMESESRVYSSAKNCTFEYVFQPFLMDLEGKQGNFTNLREFVFKNIDGYFKIYSKHTPIIVRDLPQGFSALEPLVDLPIGINITRFQTLLHRSYLTSGDSSSGWTAGAAAYYVGYLQPRTFLLKYNENGTITDAVDCALDPLSETKCTLKSFTVEKGIYQTSNFRVQPTESVVRFPNITNLCPFYEVFNATRFPSVYAWNRKRISNCVADYSVLYNFAPFFTFKCYGVSPTKLNDLCFTNVYADSFVIKGNEVSQIAPGQTGNIADYNYKLPDDFTGCVISWNSNRLDSKADGNYNYWYRLFRKSKLKPFERDISTEIYQAGNNPCNGVKGFNCYFPLQSYSFRPTYGVGYQPYRVVVLSFELLHAPATVCGPKQSTNLVKNKCVNFNFNGLTGTGVLTDSNKKFLPFQQFGRDIADTTDAVRDPQTLDILDITPCSFGGVSVITPGTNTSNQVAVLYQGVNCTEVPVAIRADQLTPTWRVYSTGSKGFQTRAGCLIGAKYVNNSYECDIPIGAGICASYQTQTRSRGSASSVDSQSIIAYTMSLGAENLVAYSNNSIAIPTNFTISVTTEILPVSMTKTSVDCTMYICGDSTECSNLLLQYGSFCTQLKRALTGIAVEQDKNTQEVFAQVKQIYKTPPITYFGGFNFSQILPDPSKPSKRSPIEDLLFNKVTLADAGFIKQYGDCLGDIAARDLICKQKFNGLTVLPPLLTDEMIAQYTSALLAGTITSGWTFGAGPALQIPFPMQMAYRFNGIGVTQNVLYENQKLIANQFNSAIGKIQDSLFSTPSALGKLQDVVNHNAQALNTLVKQLSSKFGAISSVLNDILSRLDPPEAEVQIDRLITGRLQSLQTYVTQQLIRAAEIRASANLAATKMSECVLGQSKRVDFCGKGYHLMSFPQSAPHGVVFLHVTYVPAQEKNFTTAPAICHDGKAHFPREGVFVSNGTHWFVTQRNFYEPQIITTDNTFVSGNCDVVIGIVNNTVYDPLQPELDSFKEELDKYFKNHTSRDVDLGDISGINASVVNIQKEIERLNEVAKNLNESLIDLQELGKYEQGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGRSLEVLFQGPGHHHHHHHHSAWSHPQFEKGGGSGGGGSGGSAWSHPQFEK,
22,9VBD_E1,9VBD,E1,gt3.0_le4.0,15,893,1,LTNGLDASALNALKSEWTTPPDGWEGSDPCGTNWVGITCQNDRVVSISLGNLDLEGKLPADISFLSELRILDLSYNPKLSGPLPPNIGNLGKLRNLILVGCSFSGQIPESIGTLKELIYLSLNLNKFSGTIPPSIGLLSKLYWFDIADNQIEGELPVSNGTSAPGLDMLLQTKHFHFGKNKLSGNIPKELFSSNMSLIHVLFDGNQFTGEIPETLSLVKTLTVLRLDRNKLIGDIPSYLNNLTNLNELYLANNRFTGTLPNLTSLTSLYTLDVSNNTLDFSPIPSWISSLPSLSTLRMEGIQLNGPIPISFFSPPQLQTVILKRNSIVESLDFGTDVSSQLEFVDLQYNEITDYKPSANKVLQVILANNPVCLEAGNGPSYCSAIQHNTSFSTLPTNCSPCEPGMEASPTCRCAYPFMGTLYFRSPSFSGLFNSTNFSILQKAIADFFKKFNYPVDSVGVRNIRENPTDHQLLIDLLVFPLGRESFNQTGMSLVGFAFSNQTYKPPPIFGPYIFKADLYKQFSGSAMKIEEGKLVIWINGDKGYNGLAEVGKKFEKDTGIKVTVEHPDKLEEKFPQVAATGDGPDIIFWAHDRFGGYAQSGLLAEITPDKAFQDKLYPFTWDAVRYNGKLIAYPIAVEALSLIYNKDLLPNPPKTWEEIPALDKELKAKGKSALMFNLQEPYFTWPLIAADGGYAFKYENGKYDIKDVGVDNAGAKAGLTFLVDLIKNKHMNADTDYSIAEAAFNKGETAMTINGPWAWSNIDTSKVNYGVTVLPTFKGQPSKPFVGVLSAGINAASPNKELAKEFLENYLLTDEGLEAVNKDKPLGAVALKSYEEELVKDPRIAATMENAQKGEIMPNIPQMSAFWYAVRTAVINAASGRQTVDEALKDAQT,
23,9RWS_E1,9RWS,E1,gt3.0_le4.0,16,551,1,MQPQPSATPDRKQDNKNKNYEKAGLLGFIGLLLVVAVYVAFTGNDVNWGALVFMFISYAIIFYIGSVTAGKKSDSASDMMVAGRSMPLWVAMFTMTATWVGGGYIAGTAETTFASGVVWAQAPWGYGLSLIIGGIFYARKMRRYEFTTMLDPLEVRFGKKVAGVLYLPALLGELFWSAAILVALGTTFGLILGLDFTTSIIISAIIAVAYTFVGGMWSVALTDVAQIIMIIIGMFLVVPFALSEVGGLGQAWGAYKEGMAGFTNLLPPLDGWNHPDWGNYYWNWWDYALLLIFGGIPWQVYFQRVLSAKNEKTAMWLSIAAGFLCIILAVPAVMIGVAGFSADWASYGIEGPGAASEILAYVVNYMSPYLVAIVALGAIAAAVMSSMDSSILSASSMAAWNIYRPLVKPKANGRDIKRTIRVSIILIGLAATIVALNIDSVYTLWYLCADLVYCMLFPQLTTALFDKKANTYGAIAGLSVSLFLRLGGGEPALGLPAFLPYPMIEDGVVLFPFRTLAMVSGLLTIIVVSRLTQKSCPPQPLRKLKTEMQDE,
24,9WTX_E1,9WTX,E1,gt3.0_le4.0,17,551,1,MRDYDEVIAFLGEWGPFQRLIFFLLSASIIPNGFNGMSVVFLAGTPEHRCRVPDAANLSSAWRNNSVPLRLRDGREVPHSCSRYRLATIANFSALGLEPGRDVDLGQLEQESCLDGWEFSQDVYLSTVVTEWNLVCEDNWKVPLTTSLFFVGVLLGSFVSGQLSDRFGRKNVLFATMAVQTGFSFLQIFSISWEMFTVLFVIVGMGQISNYVVAFILGTEILGKSVRIIFSTLGVCTFFAVGYMLLPLFAYFIRDWRMLLLALTVPGVLCVPLWWFIPESPRWLISQRRFREAEDIIQKAAKMNNIAVPAVIFDSVEELNPLKQQKAFILDLFRTRNIAIMTIMSLLLWMLTSVGYFALSLDAPNLHGDAYLNCFLSALIEIPAYITAWLLLRTLPRRYIIAAVLFWGGGVLLFIQLVPVDYYFLSIGLVMLGKFGITSAFSMLYVFTAELYPTLVRNMAVGVTSTASRVGSIIAPYFVYLGAYNRMLPYIVMGSLTVLIGILTLFFPESLGMTLPETLEQMQKVKWFRSGKKTRDSMETEENPKVLITAF,
25,9Z24_E1,9Z24,E1,gt3.0_le4.0,18,1214,1,MVVPEKEQSWIPKIFKKKTCTTFIVDSTDPGGTLCQCGRPRTAHPAVAMEDAFGAAVVTVWDSDAHTTEKPTDAYGELDFTGAGRKHSNFLRLSDRTDPAAVYSLVTRTWGFRAPNLVVSVLGGSGGPVLQTWLQDLLRRGLVRAAQSTGAWIVTGGLHTGIGRHVGVAVRDHQMASTGGTKVVAMGVAPWGVVRNRDTLINPKGSFPARYRWRGDPEDGVQFPLDYNYSAFFLVDDGTHGCLGGENRFRLRLESYISQQKTGVGGTGIDIPVLLLLIDGDEKMLTRIENATQAQLPCLLVAGSGGAADCLAETLEDTLAPGSGGARQGEARDRIRRFFPKGDLEVLQAQVERIMTRKELLTVYSSEDGSEEFETIVLKALVKACGSSEASAYLDELRLAVAWNRVDIAQSELFRGDIQWRSFHLEASLMDALLNDRPEFVRLLISHGLSLGHFLTPMRLAQLYSAAPSNSLIRNLLDQASHSAGTKAPALKGGAAELRPPDVGHVLRMLLGKMCAPRYPSGGAWDPHPGQGFGESMYLLSDKATSPLSLDAGLGQAPWSDLLLWALLLNRAQMAMYFWEMGSNAVSSALGACLLLRVMARLEPDAEEAARRKDLAFKFEGMGVDLFGECYRSSEVRAARLLLRRCPLWGDATCLQLAMQADARAFFAQDGVQSLLTQKWWGDMASTTPIWALVLAFFCPPLIYTRLITFRKSEEEPTREELEFDMDSVINGEGPVGTADPAEKTPLGVPRQSGRPGCCGGRCGGRRCLRRWFHFWGAPVTIFMGNVVSYLLFLLLFSRVLLVDFQPAPPGSLELLLYFWAFTLLCEELRQGLSGGGGSLASGGPGPGHASLSQRLRLYLADSWNQCDLVALTCFLLGVGCRLTPGLYHLGRTVLCIDFMVFTVRLLHIFTVNKQLGPKIVIVSKMMKDVFFFLFFLGVWLVAYGVATEGLLRPRDSDFPSILRRVFYRPYLQIFGQIPQEDMDVALMEHSNCSSEPGFWAHPPGAQAGTCVSQYANWLVVLLLVIFLLVANILLVNLLIAMFSYTFGKVQGNSDLYWKAQRYRLIREFHSRPALAPPFIVISHLRLLLRQLCRRPRSPQPSSPALEHFRVYLSKEAERKLLTWESVHKENFLLARARDKRESDSERLKRTSQKVDLALKQLGHIREYEQRLKVLEREVQQCSRVLGWVAEALSRSALLPPGGPPPPDLPGSKD,
26,26DK_E1,26DK,E1,gt3.0_le4.0,19,197,1,MTCTIDLQILQGKTFEFAFRWADPIKLYKPITGATATAPVVLTVPGHGLSDGWPFEVQSVKAPPELNTQPGQFHLAKVLTPDSLELNDVNGIGFKAFGGAGTIVFYTPADLTGLAARFAIRRTPGDADALLTGNTTDGRVVIDIPTSTISMLIGADVTATLDWNKALYDLELYDPTDVSVVYPVGSGRVTVTAEVAK,


In [ ]:
%%writefile data/entries_2026.csv
emdb_id,pdb_id,emdb_release_date,pdb_release_date,resolution_A,stratum,modeled_residues,unique_chain_seqs,unk_fraction,cc_mask,cluster_id,seed_rank,max_id_to_pre2026_pdb,cluster_size,longest_seq_len,n_chains,map_size_mb,cif_url,map_url
EMD-55586,9T5K,2026-04-22,2026-04-22,2.6,le3.0,381,1,0.0,0.8200,9T5K,1,0.614,1,453,1,250.0,https://files.rcsb.org/download/9T5K.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-55586/map/emd_55586.map.gz
EMD-68251,22FX,2026-03-04,2026-03-04,1.24,le3.0,173,1,0.0,0.7388,22FX,2,1.000,1,182,1,75.2,https://files.rcsb.org/download/22FX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-68251/map/emd_68251.map.gz
EMD-61564,9JKK,2026-04-08,2026-04-08,3.0,le3.0,530,1,0.0,0.8420,9W97,3,1.000,39,620,1,65.5,https://files.rcsb.org/download/9JKK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-61564/map/emd_61564.map.gz
EMD-56022,9TKA,2026-07-29,2026-07-29,2.7,le3.0,247,1,0.0,0.7625,9THK,4,0.000,3,293,1,182.3,https://files.rcsb.org/download/9TKA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-56022/map/emd_56022.map.gz
EMD-65930,9WFL,2026-04-01,2026-04-01,2.75,le3.0,682,2,0.0,0.8396,9WF3,5,1.000,6,697,2,65.5,https://files.rcsb.org/download/9WFL.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65930/map/emd_65930.map.gz
EMD-65077,9VI4,2026-03-11,2026-03-11,1.8,le3.0,516,1,0.0,0.8749,9P4L,6,0.984,9,516,1,1024.0,https://files.rcsb.org/download/9VI4.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65077/map/emd_65077.map.gz
EMD-74403,9ZLH,2026-09-16,2026-09-16,2.14,le3.0,206,1,0.0,0.8468,9ZK8,7,1.000,7,206,1,65.5,https://files.rcsb.org/download/9ZLH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-74403/map/emd_74403.map.gz
EMD-65363,9VUH,2026-07-15,2026-07-15,2.7,le3.0,634,3,0.0,0.7439,9VUH,8,1.000,1,365,3,128.0,https://files.rcsb.org/download/9VUH.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-65363/map/emd_65363.map.gz
EMD-70305,9OC0,2026-05-06,2026-05-06,2.97,le3.0,512,2,0.0,0.8262,9OBT,11,1.000,8,862,4,332.8,https://files.rcsb.org/download/9OC0.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70305/map/emd_70305.map.gz
EMD-76936,13BD,2026-06-17,2026-06-17,2.65,le3.0,461,1,0.0,0.8841,9OVK,12,0.954,25,1271,2,281.2,https://files.rcsb.org/download/13BD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-76936/map/emd_76936.map.gz
EMD-66160,9WQ6,2026-09-16,2026-09-16,3.4,gt3.0_le4.0,261,1,0.0,0.7629,9W3F,4,1.000,11,339,1,128.0,https://files.rcsb.org/download/9WQ6.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66160/map/emd_66160.map.gz
EMD-64777,9V4D,2026-07-01,2026-07-01,3.07,gt3.0_le4.0,276,2,0.0,0.8202,25IF,5,0.503,6,393,2,54.0,https://files.rcsb.org/download/9V4D.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64777/map/emd_64777.map.gz
EMD-70828,9OTA,2026-05-27,2026-05-27,3.2,gt3.0_le4.0,534,1,0.0,0.8452,9E94,7,1.000,2,270,2,68.7,https://files.rcsb.org/download/9OTA.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-70828/map/emd_70828.map.gz
EMD-62507,9KQM,2026-06-03,2026-06-03,3.5,gt3.0_le4.0,403,1,0.0,0.7874,9KKN,8,0.997,9,637,1,65.5,https://files.rcsb.org/download/9KQM.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-62507/map/emd_62507.map.gz
EMD-66088,9WME,2026-09-23,2026-09-23,3.2,gt3.0_le4.0,446,2,0.0,0.7050,9WME,9,1.000,1,657,2,22.8,https://files.rcsb.org/download/9WME.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66088/map/emd_66088.map.gz
EMD-64920,9VBD,2026-04-08,2026-04-08,3.26,gt3.0_le4.0,521,1,0.0,0.7685,9VBD,15,0.000,2,893,1,128.0,https://files.rcsb.org/download/9VBD.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-64920/map/emd_64920.map.gz
EMD-54346,9RWS,2026-04-29,2026-04-29,3.28,gt3.0_le4.0,512,1,0.0,0.8467,9RWS,16,0.441,1,551,1,128.0,https://files.rcsb.org/download/9RWS.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-54346/map/emd_54346.map.gz
EMD-66226,9WTX,2026-09-23,2026-09-23,3.37,gt3.0_le4.0,500,1,0.0,0.7238,9US8,17,0.763,3,551,1,128.0,https://files.rcsb.org/download/9WTX.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-66226/map/emd_66226.map.gz
EMD-73762,9Z24,2026-06-10,2026-06-10,3.1,gt3.0_le4.0,338,1,0.0,0.8749,9Z1X,18,1.000,4,1214,1,221.2,https://files.rcsb.org/download/9Z24.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-73762/map/emd_73762.map.gz
EMD-80558,26DK,2026-08-26,2026-08-26,3.5,gt3.0_le4.0,191,1,0.0,0.7407,26DK,19,0.000,2,197,1,16.0,https://files.rcsb.org/download/26DK.cif,https://ftp.ebi.ac.uk/pub/databases/emdb/structures/EMD-80558/map/emd_80558.map.gz


In [ ]:
%%writefile env/modelangelo_requirements.txt
# Exact packages of the ModelAngelo environment (Python 3.11, Linux x86_64, CUDA 12.8 wheels of PyTorch 2.9.1).
# ModelAngelo v1.0.18 (commit 994945bdfa6e5368e0d62349a47792f4864eebc3, 2026-06-15) installs 'python=3.11', 'torch==2.9.1 torchvision' and the packages of its setup.py
# (install_script.sh); psutil is imported by model_angelo/utils/misc_utils.py but is not in setup.py, so it is added. This list is what pip would pick on 2026-06-22,
# the week of that release, resolved with:
#   uv pip compile ma.in --python-version 3.11 --python-platform x86_64-manylinux_2_28 --only-binary :all: --exclude-newer 2026-06-22T00:00:00Z --no-annotate
# where ma.in holds: torch==2.9.1, torchvision, tqdm, scipy, biopython>=1.81, einops, matplotlib, mrcfile, pandas, fair-esm==1.0.3, pyhmmer==0.7.1, loguru, numpy<2.0, psutil
# The notebook installs it with 'uv pip install --no-deps -r' so that nothing else can be pulled in.
biopython==1.87
contourpy==1.3.3
cycler==0.12.1
einops==0.8.2
fair-esm==1.0.3
filelock==3.29.4
fonttools==4.63.0
fsspec==2026.6.0
jinja2==3.1.6
kiwisolver==1.5.0
loguru==0.7.3
markupsafe==3.0.3
matplotlib==3.11.0
mpmath==1.3.0
mrcfile==1.5.4
networkx==3.6.1
numpy==1.26.4
nvidia-cublas-cu12==12.8.4.1
nvidia-cuda-cupti-cu12==12.8.90
nvidia-cuda-nvrtc-cu12==12.8.93
nvidia-cuda-runtime-cu12==12.8.90
nvidia-cudnn-cu12==9.10.2.21
nvidia-cufft-cu12==11.3.3.83
nvidia-cufile-cu12==1.13.1.3
nvidia-curand-cu12==10.3.9.90
nvidia-cusolver-cu12==11.7.3.90
nvidia-cusparse-cu12==12.5.8.93
nvidia-cusparselt-cu12==0.7.1
nvidia-nccl-cu12==2.27.5
nvidia-nvjitlink-cu12==12.8.93
nvidia-nvshmem-cu12==3.3.20
nvidia-nvtx-cu12==12.8.90
packaging==26.2
pandas==3.0.3
pillow==12.2.0
psutil==5.9.8
pyhmmer==0.7.1
pyparsing==3.3.2
python-dateutil==2.9.0.post0
scipy==1.17.1
six==1.17.0
sympy==1.14.0
torch==2.9.1
torchvision==0.24.1
tqdm==4.68.3
triton==3.5.1
typing-extensions==4.15.0


In [ ]:
EXPECTED = {'scripts/nbtools.py': '9bda95fce7f7fbe0c373e315740f555fc27942a08971ce2fe1b5cae6af6d748d', 'scripts/mica_inputs.py': '87b7f5e43b86671e09e673e8bb07abb20496ad92264528dfd9994b88bded417f', 'scripts/evaluate_models.py': '8c235732d2595f8ad2014d70a278b44097205aff2417c9b5761f55dfe4974250', 'data/af3/af3_jobs.csv': 'b43068ffed7e3843d0d9c5cfa509d8f1ba9900527a1a0cad9e74aa69573cd4b0', 'data/entries_2026.csv': 'cb695bb04ef09bf48f8c2848f91a2580c0b3c15f2a8f1e580659d86f0097a5da', 'env/modelangelo_requirements.txt': 'fb1e831ccc62aaa1633b2dff7e15dd904403e493e3fe0360e95bf349d1150d4a'}
bad = [p for p, h in EXPECTED.items() if not os.path.exists(p) or sha256_file(p) != h]
if bad:
    fail("these files were not written as expected: %s. Use Run all again; if it happens again, send me this message." % ", ".join(bad))
else:
    log("all %d files written and checked (hashes match)" % len(EXPECTED))


## Step 2b. Progress lines, a zip that is refreshed, and the last cell that works after a restart

This cell adds three things that the first pilot lacked: a progress line every few minutes while a program is silent, the file `state.json`, and a zip
(`modelangelo_pilot_partial.zip` in the output files) that is written again after every stage and every ten minutes, so a stop in the middle never leaves nothing.

In [ ]:
sys.path.insert(0, os.path.join(WORK, "scripts"))
import nbtools

# Kaggle's kernel sets MPLBACKEND=module://matplotlib_inline.backend_inline. Every program started from this notebook inherits it, and `import matplotlib.pyplot`
# then fails in any environment without matplotlib-inline (ModelAngelo's, in the first pilot: the run stopped after 5 minutes). Agg needs nothing.
os.environ["MPLBACKEND"] = "Agg"

NOTEBOOK = "modelangelo_pilot"
STATE_FILE = os.path.join(WORK, "state.json")
PARTIAL_ZIP = nbtools.partial_zip_path(OUT_DIR, NOTEBOOK)
PACK_LOCK = threading.Lock()
LAST_PARTIAL = {"t": 0.0}


def save_state():
    try:
        with open(STATE_FILE, "w", encoding="utf-8") as f:
            json.dump(STATE, f, indent=1, default=str)
    except Exception as ex:
        log("could not save state.json: %s" % ex)


def pack_now(min_gap=0):
    """Writes the partial zip again (same name every time). min_gap: do nothing if the last one is younger than this many seconds."""
    with PACK_LOCK:
        if time.time() - LAST_PARTIAL["t"] < min_gap:
            return
        LAST_PARTIAL["t"] = time.time()
        try:
            save_state()
            nbtools.pack(WORK, PARTIAL_ZIP, NOTEBOOK + "_partial", secrets=SECRETS)
        except Exception as ex:
            log("the partial zip could not be written: %s" % ex)


BEAT = float(os.environ.get("MICA_BEAT", "120"))          # seconds between progress lines (the tests make it short)


def run_watched(cmd, label, watch=(), beat=None, zip_every=600, **kw):
    """run() with a progress line every `beat` seconds (time, load, free RAM, busiest processes, last line of the files in `watch`)."""
    hb = nbtools.Heartbeat(label, log, watch=watch, every=beat or BEAT, on_tick=lambda: pack_now(zip_every))
    hb.start()
    try:
        return run(cmd, **kw)
    finally:
        hb.stop()


def finish(complete, extra_info=None):
    """Writes run_info.json and the final zip; returns the path of the zip."""
    info = {"notebook": NOTEBOOK, "started_utc": STATE["started"], "finished_utc": nbtools.utc_now().isoformat(), "elapsed_minutes": round((time.time() - STATE["t0"]) / 60),
            "problem": STATE["problem"], "complete": complete, "platform": PLATFORM, "python": platform.python_version(), "os": platform.platform(), "gpu": STATE["gpu"],
            "cpu_cores": os.cpu_count(), "ram_gb": meminfo_gb(), "state": STATE}
    info.update(extra_info or {})
    with open(os.path.join(WORK, "run_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, default=str)
    save_state()
    path, n, left_out = nbtools.final_pack(WORK, OUT_DIR, NOTEBOOK, complete, secrets=SECRETS)
    log("zip written: %s (%d files, %d MB)" % (path, n, os.path.getsize(path) // 1000000))
    if left_out:
        log("left out of the zip: %s" % "; ".join(left_out))
    return path


def cpu_facts():
    """How many processors the program sees and how many it may really use (a container can show more cores than its limit allows)."""
    import multiprocessing
    facts = {"os.cpu_count": os.cpu_count(), "multiprocessing.cpu_count": multiprocessing.cpu_count(),
             "usable (sched_getaffinity)": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None, "nproc": sh("nproc 2>&1"),
             "cgroup v2 cpu.max": sh("cat /sys/fs/cgroup/cpu.max 2>&1"), "cgroup v1 quota": sh("cat /sys/fs/cgroup/cpu/cpu.cfs_quota_us 2>&1"),
             "MICA's nproc formula min(max(int(0.6 * cpu_count), 1), 48)": min(max(int(0.6 * multiprocessing.cpu_count()), 1), 48)}
    with open(os.path.join(WORK, "cpu_facts.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join("%s: %s" % kv for kv in facts.items()) + "\n")
    log("processors: " + " | ".join("%s %s" % (k.split(" ")[0], v) for k, v in facts.items()))
    STATE["cpu_facts"] = facts


save_state()
cpu_facts()
log("progress lines, state.json and the partial zip (%s) are ready" % PARTIAL_ZIP)


## Step 3. Python 3.11 environment for ModelAngelo

ModelAngelo's own install script uses Python 3.11 and `torch==2.9.1`. The notebook builds the environment with `uv` from PyPI wheels with the exact pins of
`env/modelangelo_requirements.txt` (47 packages, the versions that pip would pick in the week of the ModelAngelo release), tests the imports, and runs a small
computation on the GPU, so a GPU that this PyTorch cannot drive is found here and not later.

In [ ]:
UV_VERSION = "0.13.1"
PY_VERSION = "3.11.14"
ENV_PY_GIVEN = None                       # None: build the environment; a path is only used by the tests
ENV_DIR = os.path.join(BIG, "ma_env")
ENV_PY = ENV_PY_GIVEN or os.path.join(ENV_DIR, "bin", "python")
ENV_BIN = os.path.dirname(ENV_PY)
MA_ENV = {"PATH": ENV_BIN + os.pathsep + os.environ.get("PATH", ""), "VIRTUAL_ENV": None, "PYTHONPATH": None, "PYTHONHOME": None, "MPLBACKEND": "Agg"}
REQ_FILE = "env/modelangelo_requirements.txt"
IMPORT_TEST = "import sys, torch, numpy, scipy, pandas, Bio, mrcfile, einops, esm, pyhmmer, loguru, psutil, matplotlib, matplotlib.pyplot; print('python', sys.version.split()[0], '| torch', torch.__version__, '| cuda', torch.version.cuda, '| numpy', numpy.__version__, '| pandas', pandas.__version__, '| biopython', Bio.__version__, '| pyhmmer', pyhmmer.__version__)"
GPU_TEST = r"""
import torch
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit(3)
p = torch.cuda.get_device_properties(0)
print("gpu: %s | %.1f GB | capability %d.%d | this PyTorch is compiled for %s" % (p.name, p.total_memory / 2**30, p.major, p.minor, torch.cuda.get_arch_list()))
x = torch.randn(1, 4, 24, 24, 24, device="cuda")
y = torch.nn.Conv3d(4, 8, 3, padding=1).cuda()(x)
a = torch.randn(512, 512, device="cuda")
z = float((a @ a).sum()) + float(y.sum())
torch.cuda.synchronize()
print("3D convolution and matrix product on the GPU work")
"""


def pinned(path=REQ_FILE):
    with open(path, encoding="utf-8") as f:
        return [ln.strip() for ln in f if "==" in ln and not ln.startswith("#")]


def env_imports():
    return run([ENV_PY, "-c", IMPORT_TEST], env=MA_ENV, quiet=True)


def make_env():
    rc, last, _ = env_imports() if os.path.exists(ENV_PY) else (1, [], False)
    if rc == 0:
        log("the Python environment already exists: " + last[-1])
        return True
    tools = {"UV_CACHE_DIR": os.environ.get("UV_CACHE_DIR") or os.path.join(BIG, "uv_cache"),
             "UV_PYTHON_INSTALL_DIR": os.environ.get("UV_PYTHON_INSTALL_DIR") or os.path.join(BIG, "uv_python"), "UV_PYTHON_PREFERENCE": "only-managed"}
    rc, last, _ = run([sys.executable, "-m", "pip", "install", "-q", "uv==" + UV_VERSION])
    if rc != 0:
        fail("pip could not install uv (see the lines above). Run the notebook again in a few minutes.")
        return False
    uv = [sys.executable, "-m", "uv"]
    log("uv: " + sh("%s -m uv --version" % sys.executable))
    for what, cmd in (("Python " + PY_VERSION, uv + ["python", "install", PY_VERSION]),
                      ("the virtual environment", uv + ["venv", "--python", PY_VERSION, ENV_DIR]),
                      ("the %d packages (a few minutes; PyTorch with its CUDA libraries is about 4 GB)" % len(pinned()),
                       uv + ["pip", "install", "--python", ENV_PY, "--no-deps", "-r", REQ_FILE])):
        log("installing " + what + " ...")
        rc, last, _ = run_watched(cmd, "uv: " + what[:30], env=tools, every=20)
        if rc != 0:
            fail("uv could not install %s (exit code %d, the last lines are above). Run the notebook again in a few minutes; if it happens again, send me the zip." % (what, rc))
            return False
    rc, last, _ = env_imports()
    if rc != 0 and any("libGL" in ln or "libgomp" in ln for ln in last):
        log("a system library is missing; installing it with apt ...")
        run(["bash", "-c", "apt-get update -qq && apt-get install -y -qq libgl1 libgomp1"], every=20)
        rc, last, _ = env_imports()
    if rc != 0:
        log("\n".join(last))
        fail("the Python packages were installed but do not import (see the lines above). Send me the zip.")
        return False
    log(last[-1])
    return True


def gpu_test():
    rc, last, _ = run([ENV_PY, "-c", GPU_TEST], env=MA_ENV)
    if rc == 3:
        fail("PyTorch in the ModelAngelo environment does not see the GPU. Switch the GPU on (Kaggle: Session options -> Accelerator, GPU T4 x2) and run again.")
    elif rc != 0:
        fail("this GPU cannot run PyTorch 2.9.1 with CUDA 12.8 (see the lines above). Choose another GPU: Kaggle GPU T4 x2 (the P100 is too old for this PyTorch).")
    return rc == 0


def freeze_check():
    code = "import importlib.metadata as m; print(chr(10).join(sorted('%s==%s' % (d.metadata['Name'], d.version) for d in m.distributions())))"
    if os.path.exists("ma_env_freeze.txt"):
        os.remove("ma_env_freeze.txt")
    run([ENV_PY, "-c", code], env=MA_ENV, log_file="ma_env_freeze.txt", quiet=True)
    norm = lambda line: re.sub(r"[-_.]+", "-", line.strip().lower())
    got = {norm(ln) for ln in open("ma_env_freeze.txt", encoding="utf-8") if "==" in ln}
    want = {norm(ln) for ln in pinned()}
    differences = sorted(got ^ want)
    STATE["env"] = {"python_version": PY_VERSION, "uv_version": UV_VERSION, "packages": len(got), "differences_to_pinned_list": differences[:20]}
    log("installed packages: %d; differences to %s: %d %s" % (len(got), REQ_FILE, len(differences), differences[:6] or ""))


if STATE["ok"]:
    if make_env():
        if REQUIRE_GPU:
            gpu_test()
        if ENV_PY_GIVEN is None:
            freeze_check()
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Step 4. ModelAngelo code, weights, US-align

ModelAngelo is fetched at its pinned commit and run from that source folder (`python -m model_angelo`, the same program as the `model_angelo` command).
`model_angelo setup_weights --bundle-name nucleotides` downloads the bundle that `model_angelo build` uses by default (Zenodo, MD5 checked by ModelAngelo) and the
ESM-1b language model (Meta); `TORCH_HOME` points to a folder of this session. US-align is compiled from its pinned commit.

In [ ]:
MA_REPO = "https://github.com/3dem/model-angelo"
MA_COMMIT = "994945bdfa6e5368e0d62349a47792f4864eebc3"
MA_BUNDLE = "nucleotides"
USALIGN_REPO = "https://github.com/pylelab/USalign"
USALIGN_COMMIT = "746ad8a2a4dea9f317075cc482df0ecb5440bdea"
USALIGN_GIVEN = None                      # None: compile it; a path is only used by the tests
MA_DIR = os.path.join(BIG, "ModelAngelo")
TORCH_HOME = os.path.join(BIG, "torch_home")
MA_RUN_ENV = dict(MA_ENV, TORCH_HOME=TORCH_HOME)


def git_checkout(repo, commit, folder):
    if os.path.isdir(os.path.join(folder, ".git")) and sh("git -C %s rev-parse HEAD" % folder) == commit:
        return True
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)
    for cmd in (["git", "init", "-q", folder], ["git", "-C", folder, "remote", "add", "origin", repo],
                ["git", "-C", folder, "fetch", "-q", "--depth", "1", "origin", commit], ["git", "-C", folder, "checkout", "-q", "FETCH_HEAD"]):
        rc, last, _ = run(cmd)
        if rc != 0:
            log("\n".join(last))
            return False
    return sh("git -C %s rev-parse HEAD" % folder) == commit


def build_usalign():
    if USALIGN_GIVEN:
        STATE["usalign"] = USALIGN_GIVEN
    else:
        src = os.path.join(BIG, "USalign_src")
        exe = os.path.join(src, "USalign")
        if not os.path.exists(exe):
            if not git_checkout(USALIGN_REPO, USALIGN_COMMIT, src):
                fail("US-align could not be fetched at commit %s" % USALIGN_COMMIT)
                return False
            rc, last, _ = run(["g++", "-O3", "-ffast-math", "-o", exe, os.path.join(src, "USalign.cpp"), "-lm"], cwd=src, every=10)
            if rc != 0:
                fail("US-align could not be compiled (see the lines above). Send me the zip.")
                return False
        STATE["usalign"] = exe
    r = subprocess.run([STATE["usalign"], "-h"], capture_output=True, text=True, errors="replace")
    head = [ln for ln in (r.stdout + r.stderr).splitlines() if "Version" in ln]
    STATE["usalign_version"] = head[0].strip() if head else "unknown"
    log("US-align: %s (%s)" % (STATE["usalign"], STATE["usalign_version"]))
    return True


def fetch_modelangelo():
    if not git_checkout(MA_REPO, MA_COMMIT, MA_DIR):
        fail("ModelAngelo could not be fetched at commit %s (see the lines above). Run the notebook again in a few minutes." % MA_COMMIT)
        return False
    missing = [p for p in ("model_angelo/__main__.py", "model_angelo/model_definitions/" + MA_BUNDLE) if not os.path.exists(os.path.join(MA_DIR, p))]
    if missing:
        fail("the ModelAngelo source at commit %s lacks %s" % (MA_COMMIT, ", ".join(missing)))
        return False
    log("ModelAngelo at commit %s" % MA_COMMIT)
    rc, last, _ = run([ENV_PY, "-m", "model_angelo", "--version"], env=MA_RUN_ENV, cwd=MA_DIR, quiet=True)
    STATE["modelangelo_version"] = (last or [""])[-1]
    log("version printed by the program: %s" % STATE["modelangelo_version"])
    code = "import model_angelo.apps.build, model_angelo.utils.setup_weights; print('ModelAngelo modules import')"
    rc, last, _ = run([ENV_PY, "-c", code], env=MA_RUN_ENV, cwd=MA_DIR)
    if rc != 0:
        fail("ModelAngelo's own Python modules cannot be imported (see the lines above). Send me the zip.")
        return False
    return True


def bundle_dir():
    return os.path.join(TORCH_HOME, "hub", "checkpoints", "model_angelo_v1.0", MA_BUNDLE)


def fetch_weights():
    logf = os.path.join(WORK, "logs", "setup_weights.log")
    cmd = [ENV_PY, "-m", "model_angelo", "setup_weights", "--bundle-name", MA_BUNDLE]
    log("downloading the weights: %s" % " ".join(cmd[1:]))
    t0 = time.time()
    rc, last, timed_out = run_watched(cmd, "weights download", watch=[logf], env=MA_RUN_ENV, cwd=MA_DIR, log_file=logf, timeout=3600, every=1)
    text = open(logf, errors="replace").read() if os.path.exists(logf) else ""
    if rc != 0 or timed_out or "Successful!" not in text:
        log("\n".join(last[-15:]))
        fail("the ModelAngelo weights could not be downloaded or did not pass ModelAngelo's own MD5 check (%s). Run the notebook again in a few minutes; if it happens again, send me the zip."
             % ("timeout" if timed_out else "exit code %s" % rc))
        return False
    files = {}
    for dirpath, _dirs, names in os.walk(TORCH_HOME):
        for n in names:
            p = os.path.join(dirpath, n)
            if n.endswith((".torch", ".pt", ".json", ".txt", ".zip")) and os.path.getsize(p) > 0:
                files[os.path.relpath(p, TORCH_HOME)] = os.path.getsize(p)
    cfg = os.path.join(bundle_dir(), "config.json")
    if not os.path.exists(cfg):
        fail("the weights bundle %s was not installed where expected (%s)" % (MA_BUNDLE, bundle_dir()))
        return False
    shutil.copy(cfg, os.path.join(WORK, "modelangelo_config.json"))
    STATE["weights"] = {"bundle": MA_BUNDLE, "seconds": round(time.time() - t0), "md5_checked_by_modelangelo": "Checksum of downloaded file matches expected checksum" in text,
                        "files_bytes": files, "bundle_dir": bundle_dir()}
    log("weights installed in %d s; files: %s" % (STATE["weights"]["seconds"], {k: v for k, v in files.items() if v > 1000000}))
    return True


if STATE["ok"]:
    if fetch_modelangelo() and fetch_weights():
        build_usalign()
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Step 5. The entry: map, deposited model, FASTA file

The map comes from EMDB (unzipped to a `.mrc` file), the deposited model from RCSB (only used for scoring), and the FASTA file from the job table of the benchmark
(`data/af3/af3_jobs.csv`: one record per different chain sequence, with the number of copies in the header, the format of the RCSB FASTA files that ModelAngelo reads).
Every method of the benchmark gets this same sequence information.

In [ ]:
sys.path.insert(0, os.path.join(WORK, "scripts"))
import evaluate_models, mica_inputs

ENTRIES = ['26DK']
INPUT_DIR = os.path.join(BIG, "input")
METRIC_COLUMNS = ["entry", "model_kind", "model_file", "status", "note", "tm_score", "tm_score_pred_norm", "ca_rmsd", "aligned_ca_length", "seq_identity",
                  "pred_residues", "native_residues", "coverage"]


def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def write_csv(path, columns, rows):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns, lineterminator="\n", extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def download(url, dest, min_bytes=1, tries=3):
    import requests
    if os.path.exists(dest) and os.path.getsize(dest) >= min_bytes:
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    last = ""
    for attempt in range(1, tries + 1):
        try:
            with requests.get(url, stream=True, timeout=(30, 120)) as r:
                r.raise_for_status()
                done = 0
                with open(dest + ".part", "wb") as f:
                    for block in r.iter_content(1 << 20):
                        f.write(block)
                        done += len(block)
            if done < min_bytes:
                raise IOError("only %d bytes arrived" % done)
            os.replace(dest + ".part", dest)
            return dest
        except (requests.RequestException, IOError) as ex:
            last = str(ex)[:200]
            log("download attempt %d of %d failed (%s): %s" % (attempt, tries, url, last))
            time.sleep(5 * attempt)
    raise RuntimeError("could not download %s: %s" % (url, last))


def prepare_entry(entry, row, res):
    folder = os.path.join(INPUT_DIR, entry)
    os.makedirs(folder, exist_ok=True)
    number = row["emdb_id"].split("-")[-1]
    map_path = os.path.join(folder, "emd_%s.mrc" % number)
    if not os.path.exists(map_path):
        gz = download(row["map_url"], os.path.join(BIG, "downloads", os.path.basename(row["map_url"])), 100000)
        try:
            with gzip.open(gz, "rb") as src, open(map_path, "wb") as out:
                shutil.copyfileobj(src, out)
        except OSError:
            shutil.copyfile(gz, map_path)                           # not gzipped after all
    res["map_sha256"] = sha256_file(map_path)
    run([ENV_PY, "-c", "import sys, mrcfile; m = mrcfile.open(sys.argv[1], permissive=True); d = m.data; "
         "print('map grid', d.shape, 'voxel size', [round(float(v), 4) for v in (m.voxel_size.x, m.voxel_size.y, m.voxel_size.z)], "
         "'min %.4g max %.4g mean %.4g std %.4g' % (d.min(), d.max(), d.mean(), d.std()))", map_path], env=MA_ENV)
    native = download(row["cif_url"], os.path.join(WORK, "native", entry + ".cif"), 1000)
    jobs = mica_inputs.read_jobs("data/af3/af3_jobs.csv", entry)
    fasta = os.path.join(folder, entry.lower() + ".fasta")
    with open(fasta, "w", encoding="utf-8") as f:
        f.write(mica_inputs.mica_fasta(jobs))
    os.makedirs(os.path.join(WORK, "inputs"), exist_ok=True)
    shutil.copy(fasta, os.path.join(WORK, "inputs", os.path.basename(fasta)))
    res["sequences"] = [{"job": r["job_name"], "length": int(r["length"]), "copies": int(r["copies_in_entry"])} for r in jobs]
    log("%s: map %s (sha256 %s), FASTA with %d sequence(s) %s" % (entry, os.path.basename(map_path), res["map_sha256"][:16], len(jobs), res["sequences"]))
    return map_path, fasta, native


## Step 6. Run ModelAngelo

`model_angelo build -v <map> -pf <fasta> -o <folder> --device 0`, the command of ModelAngelo's README with the GPU named explicitly. A progress line is printed every two
minutes with the last line of ModelAngelo's own log; GPU memory and free RAM are sampled every 5 seconds into `gpu/`. The time limit is 3 hours; a stopped run is a
result and stays in the tables.

In [ ]:
OUT_ROOT = os.path.join(WORK, "ma_out")
TIMEOUT = 3 * 3600                  # seconds; a run that takes longer is stopped and counted as failed
ENTRY_RESULTS = {}


def run_modelangelo(entry, map_path, fasta, res):
    out_dir = os.path.join(OUT_ROOT, "%s_modelangelo" % entry)     # ModelAngelo names its result after this folder
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(OUT_ROOT, exist_ok=True)
    cmd = [ENV_PY, "-m", "model_angelo", "build", "-v", map_path, "-pf", fasta, "-o", out_dir, "--device", "0"]
    log("\n=== ModelAngelo: %s" % " ".join(cmd[1:]))
    logf = os.path.join(WORK, "logs", "%s_modelangelo.log" % entry)
    mon = Monitor(os.path.join(WORK, "gpu", "%s_modelangelo.csv" % entry))
    mon.start()
    t0 = time.time()
    rc, last, timed_out = run_watched(cmd, "ModelAngelo", watch=[logf, os.path.join(out_dir, "model_angelo.log")], env=MA_RUN_ENV, cwd=MA_DIR, log_file=logf, timeout=TIMEOUT, every=1)
    info = mon.stop()
    info.update(rc=rc, seconds=round(time.time() - t0), timed_out=timed_out)
    res["steps"]["modelangelo"] = info
    log("ModelAngelo finished: exit code %s, %d s, GPU memory peak %s MB, lowest free RAM %s GB" % (rc, info["seconds"], info["peak_gpu_mb"], info["min_ram_available_gb"]))
    final, raw = os.path.join(out_dir, "%s_modelangelo.cif" % entry), os.path.join(out_dir, "%s_modelangelo_raw.cif" % entry)
    made = os.path.exists(final) and os.path.getsize(final) > 500
    if rc != 0 or timed_out or not made:
        res.update(failed_step="modelangelo", note=("timeout" if timed_out else "exit code %s" % rc) + ("" if made else "; the model file %s was not written" % os.path.basename(final)))
        log("ENTRY %s STOPPED in ModelAngelo: %s. The last lines of the program:\n%s" % (entry, res["note"], "\n".join(last[-20:])))
        return None
    log("ModelAngelo wrote: %s" % ", ".join("%s (%d bytes)" % (os.path.relpath(os.path.join(d, n), out_dir), os.path.getsize(os.path.join(d, n))) for d, _s, ns in os.walk(out_dir) for n in sorted(ns)))
    return {"modelangelo": final, "modelangelo_raw": raw}


def score_entry(entry, row, native, models, res):
    rows = []
    os.makedirs(os.path.join(WORK, "scoring"), exist_ok=True)
    for kind, path in models.items():
        if not os.path.exists(path):
            log("%s: %s was not written, not scored" % (entry, os.path.basename(path)))
            continue
        ev = evaluate_models.evaluate(STATE["usalign"], path, native, int(row["modeled_residues"]))
        full = subprocess.run([STATE["usalign"], path, native, "-mol", "protein"], capture_output=True, text=True, errors="replace")
        with open(os.path.join(WORK, "scoring", "%s_%s_usalign.txt" % (entry, kind)), "w", encoding="utf-8") as f:
            f.write("# USalign %s %s -mol protein\n%s%s" % (path, native, full.stdout, full.stderr))
        rows.append(dict(ev, entry=entry, model_kind=kind, model_file=os.path.basename(path)))
    res["models"] = rows
    return rows


def process_entry(entry):
    res = {"entry": entry, "status": "failed", "failed_step": "", "note": "", "steps": {}, "models": []}
    STATE["entries"][entry] = res
    t0 = time.time()
    row = {r["pdb_id"]: r for r in read_csv("data/entries_2026.csv")}.get(entry)
    if row is None:
        res.update(failed_step="inputs", note="%s is not in data/entries_2026.csv" % entry)
        return res
    log("\n" + "=" * 72 + "\nENTRY %s (EMDB %s, %s A, %s modeled residues)\n" % (entry, row["emdb_id"], row["resolution_A"], row["modeled_residues"]) + "=" * 72)
    try:
        map_path, fasta, native = prepare_entry(entry, row, res)
    except Exception as ex:
        res.update(failed_step="inputs", note=str(ex)[:500])
        log("\nENTRY %s STOPPED while preparing the inputs: %s" % (entry, ex))
        return res
    pack_now()
    models = run_modelangelo(entry, map_path, fasta, res)
    ok = models is not None
    pack_now()
    if ok:
        try:
            score_entry(entry, row, native, models, res)
        except Exception as ex:
            res.update(failed_step="scoring", note=("scoring: %s" % ex)[:500])
            log("scoring stopped: %s" % ex)
            ok = False
    if ok:
        res["status"] = "ok"
    res["seconds"] = round(time.time() - t0)
    return res


def summary_table():
    log("\n%-5s %-16s %-8s %7s %8s %7s %9s %8s" % ("entry", "model", "status", "TM", "aligned", "RMSD", "coverage", "seq id"))
    for e in STATE["entries"].values():
        for m in e["models"]:
            log("%-5s %-16s %-8s %7s %8s %7s %9s %8s" % (m["entry"], m["model_kind"], m["status"], m["tm_score"], m["aligned_ca_length"], m["ca_rmsd"], m["coverage"], m["seq_identity"]))
        log("%-5s ENTRY %s%s; seconds: %s" % (e["entry"], e["status"].upper(), (" (stopped in %s: %s)" % (e["failed_step"], e["note"][:120])) if e["failed_step"] else "",
                                             {k: v["seconds"] for k, v in e["steps"].items()}))


if STATE["ok"] and STATE["usalign"]:
    for entry in ENTRIES:
        try:
            process_entry(entry)
        except Exception:
            import traceback
            STATE["entries"].setdefault(entry, {"entry": entry, "steps": {}, "models": []}).update(status="failed", failed_step="unexpected error", note=traceback.format_exc()[-600:])
            log("\nENTRY %s STOPPED by an unexpected error:\n%s" % (entry, traceback.format_exc()))
    write_csv("pilot_metrics.csv", METRIC_COLUMNS, [m for e in STATE["entries"].values() for m in e["models"]])
    write_csv("entry_status.csv", ["entry", "status", "failed_step", "note", "seconds", "map_sha256"], list(STATE["entries"].values()))
    summary_table()
    STATE["complete"] = bool(STATE["entries"]) and all(e["status"] == "ok" for e in STATE["entries"].values())
    if not STATE["complete"]:
        STATE["stopped_at"] = ", ".join("%s: %s" % (e["entry"], e["failed_step"]) for e in STATE["entries"].values() if e["status"] != "ok")
    pack_now()
else:
    log("skipped: an earlier step stopped")


## Last step. Pack the results

Run this cell last, also after a problem. It works after a restart of the kernel too: it then packs the files that are on disk.

In [ ]:
import os, sys
try:
    finish
except NameError:                                  # fresh kernel: nothing of the earlier cells is left, only the files on disk
    work = None
    for cand in (os.environ.get("MICA_WORK"), "/kaggle/working/pilot", "/content/pilot", os.path.abspath("pilot")):
        if cand and os.path.isfile(os.path.join(cand, "scripts", "nbtools.py")):
            work = cand
            break
    if work is None:
        print("RECOVERY: the work folder was not found, so there is nothing to pack. Run the notebook from the top.")
    else:
        sys.path.insert(0, os.path.join(work, "scripts"))
        import nbtools
        nbtools.recover("modelangelo_pilot")
else:
    zip_path = finish(bool(STATE["ok"] and STATE.get("complete")), {"modelangelo_commit": globals().get("MA_COMMIT", ""), "modelangelo_version": STATE.get("modelangelo_version", ""), "bundle": globals().get("MA_BUNDLE", ""), "weights": STATE.get("weights", {}), "env": STATE["env"], "usalign_commit": globals().get("USALIGN_COMMIT", ""), "usalign_version": STATE["usalign_version"], "input_sha256": globals().get("EXPECTED", {})})
    if STATE["ok"] and STATE.get("complete"):
        log("\nRESULT: OK. Download %s (Kaggle: the output files of this version) and send it to me." % zip_path)
    elif STATE["ok"]:
        log("\nRESULT: NOT COMPLETE (%s). Send me %s; it shows where it stopped." % (STATE.get("stopped_at") or "a step did not finish", zip_path))
    else:
        log("\nRESULT: STOPPED (%s). Send me %s." % (STATE["problem"], zip_path))
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception:
        log("(Kaggle: open the version, then Output, then download the zip. Colab: the file list on the left.)")
